**若需使用bsae INT8模型，第一次运行请连接CPU，先做模型量化，量化模型保存于google driver。**

In [ ]:
# @title 挂载Google硬盘
%%capture
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title z-image-base-INT8量化 （连接CPU,只需运行一次，今后使用不再运行）

# ==========================================
# 步骤 1：安装依赖
# ==========================================
!pip install -U -q git+https://github.com/Disty0/sdnq git+https://github.com/huggingface/diffusers
!pip install -U -q transformers accelerate safetensors psutil

# ==========================================
# 步骤 2：主量化流程
# ==========================================
import os
import json
import gc
import shutil
import torch
from google.colab import drive
from safetensors.torch import save_file, load_file
from huggingface_hub import hf_hub_download
import psutil

def print_mem():
    mem = psutil.virtual_memory()
    used_gb = mem.used / 1e9
    total_gb = mem.total / 1e9
    percent = mem.percent
    print(f"💻 RAM: {used_gb:.1f}GB / {total_gb:.1f}GB ({percent:.1f}%)")
    if percent > 85:
        print("⚠️  内存使用超过 85%，强制 GC")
        gc.collect()

def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 挂载 Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

FINAL_DIR = "/content/drive/MyDrive/z-image-base-transformer-int8"
LOCAL_DIR = "/content/quant_output"
TEMP_DIR = "/content/temp_quant"

for d in [LOCAL_DIR, TEMP_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

# 与 Turbo 完全一致的排除列表
target_modules_to_not_convert = [
    "layers.0.adaLN_modulation.0.weight",
    "all_x_embedder",
    "t_embedder",
    "cap_embedder",
    "all_final_layer"
]

def should_quantize(name):
    """判断是否量化（与 Turbo 逻辑一致）"""
    if name in target_modules_to_not_convert:
        return False

    for skip_prefix in target_modules_to_not_convert:
        if name.startswith(skip_prefix):
            return False

    return name.endswith(".weight")

def quantize_tensor_int8_per_channel(tensor):
    """
    INT8 Per-Channel 对称量化（与 Turbo 完全对齐）

    对于 2D 权重 [out_features, in_features]：
    - 对每个输出通道（dim=0）独立计算 scale
    - 返回 scale shape: [out_features, 1]（与 Turbo 一致）

    对于 1D 权重（如 bias/norm）：
    - 使用 per-tensor 量化
    """
    if tensor.dtype == torch.int8:
        return tensor, None

    t_float = tensor.float()

    # 🔧 关键：2D 权重用 per-channel，1D 用 per-tensor
    if tensor.dim() == 2:
        # Per-Channel 量化
        # 沿 dim=1 求最大值，保留 dim=0（输出通道）
        abs_max_per_channel = t_float.abs().amax(dim=1, keepdim=True)  # shape: [out_features, 1]

        # 避免除零
        abs_max_per_channel = torch.clamp(abs_max_per_channel, min=1e-8)

        # 计算 scale（每个输出通道一个）
        scale = abs_max_per_channel / 127.0  # shape: [out_features, 1]

        # 量化（自动广播）
        q_tensor = (t_float / scale).round().clamp(-128, 127).to(torch.int8)

        # 🔧 关键：保持 scale 的 shape 为 [out_features, 1]（与 Turbo 一致）
        return q_tensor, scale.to(torch.bfloat16)

    else:
        # Per-Tensor 量化（用于 1D/3D+ 权重）
        abs_max = t_float.abs().max().item()

        if abs_max == 0 or abs_max < 1e-8:
            return tensor.to(torch.int8), torch.tensor(1.0, dtype=torch.bfloat16)

        scale = abs_max / 127.0
        q_tensor = (t_float / scale).round().clamp(-128, 127).to(torch.int8)

        # 🔧 返回标量 scale
        return q_tensor, torch.tensor(scale, dtype=torch.bfloat16)

print("="*60)
print("🚀 开始 Per-Channel 量化流程（Turbo 完全对齐）")
print("="*60)
print_mem()

# ==========================================
# 1. 下载原始模型
# ==========================================
print("\n📥 步骤 1: 下载模型索引...")

index_file = hf_hub_download(
    "Tongyi-MAI/Z-Image",
    "transformer/diffusion_pytorch_model.safetensors.index.json",
    cache_dir=TEMP_DIR,
    resume_download=True
)

with open(index_file, "r") as f:
    original_index = json.load(f)

weight_map = original_index["weight_map"]
shard_files = sorted(set(weight_map.values()))

print(f"   ✓ 原始分片数: {len(shard_files)}")
print(f"   ✓ 总参数数: {len(weight_map)}")

config_file = hf_hub_download(
    "Tongyi-MAI/Z-Image",
    "transformer/config.json",
    cache_dir=TEMP_DIR,
    resume_download=True
)

with open(config_file, "r") as f:
    base_config = json.load(f)

print(f"   ✓ config.json 已下载")
print_mem()

# ==========================================
# 2. 逐分片量化
# ==========================================
print(f"\n🔄 步骤 2: 逐分片量化...")

TEMP_SHARD_SIZE = 500_000_000  # 500MB

temp_shard_idx = 1
current_temp_shard = {}
current_temp_size = 0
scales_dict = {}

quantized_count = 0
kept_count = 0

for shard_idx, shard_name in enumerate(shard_files):
    print(f"\n{'─'*60}")
    print(f"📦 分片 {shard_idx+1}/{len(shard_files)}: {shard_name}")
    print_mem()

    shard_path = hf_hub_download(
        "Tongyi-MAI/Z-Image",
        f"transformer/{shard_name}",
        cache_dir=TEMP_DIR,
        resume_download=True
    )

    shard_data = load_file(shard_path)
    print(f"   包含参数: {len(shard_data)} 个")

    for param_idx, (param_name, tensor) in enumerate(shard_data.items()):
        tensor = tensor.cpu()

        if should_quantize(param_name) and tensor.dim() >= 2:
            # 🔧 使用 Per-Channel 量化
            q_tensor, scale = quantize_tensor_int8_per_channel(tensor)

            if scale is not None:
                # 🔧 关键：scale 键名格式
                scale_key = param_name.replace(".weight", ".scale")
                scales_dict[scale_key] = scale

                quantized_count += 1

                if param_idx < 3:
                    print(f"   ✓ [{param_idx+1}] {param_name}")
                    print(f"      量化: {list(tensor.shape)} → int8")
                    print(f"      scale shape: {list(scale.shape)} (per-channel)")

        else:
            # 保持为 bfloat16
            q_tensor = tensor.to(torch.bfloat16) if tensor.dtype != torch.bfloat16 else tensor
            kept_count += 1

            if param_idx < 3:
                print(f"   - [{param_idx+1}] {param_name}")
                print(f"      保持: bfloat16")

        param_size = q_tensor.numel() * q_tensor.element_size()

        # 内存控制：达到临时分片大小就保存
        if current_temp_size + param_size > TEMP_SHARD_SIZE and current_temp_shard:
            temp_file = os.path.join(TEMP_DIR, f"temp_{temp_shard_idx:04d}.safetensors")
            print(f"\n   💾 暂存临时分片 #{temp_shard_idx} ({len(current_temp_shard)} 参数)")
            save_file(current_temp_shard, temp_file)

            current_temp_shard.clear()
            current_temp_size = 0
            temp_shard_idx += 1
            force_cleanup()

        current_temp_shard[param_name] = q_tensor
        current_temp_size += param_size

        del tensor, q_tensor

    del shard_data
    force_cleanup()

    try:
        os.remove(shard_path)
    except:
        pass

# 保存最后一个临时分片
if current_temp_shard:
    temp_file = os.path.join(TEMP_DIR, f"temp_{temp_shard_idx:04d}.safetensors")
    print(f"\n💾 保存最后临时分片 #{temp_shard_idx}")
    save_file(current_temp_shard, temp_file)
    del current_temp_shard
    force_cleanup()

total_temp_shards = temp_shard_idx

# 保存 scales
if scales_dict:
    print(f"\n💾 保存量化 scales: {len(scales_dict)} 个")

    # 🔧 验证：检查 scale 的 shape
    sample_scale_keys = list(scales_dict.keys())[:3]
    print(f"   前 3 个 scale 的 shape:")
    for k in sample_scale_keys:
        print(f"      {k}: {list(scales_dict[k].shape)}")

    save_file(scales_dict, os.path.join(TEMP_DIR, "temp_scales.safetensors"))
    del scales_dict
    force_cleanup()

print(f"\n{'='*60}")
print(f"✅ 量化统计:")
print(f"   量化参数: {quantized_count}")
print(f"   保持参数: {kept_count}")
print(f"   临时分片数: {total_temp_shards}")
print(f"{'='*60}")
print_mem()

# ==========================================
# 3. 合并为单个文件
# ==========================================
print(f"\n🔗 步骤 3: 合并为单个文件...")

final_state_dict = {}

for i in range(1, total_temp_shards + 1):
    temp_file = os.path.join(TEMP_DIR, f"temp_{i:04d}.safetensors")

    if not os.path.exists(temp_file):
        continue

    print(f"   [{i}/{total_temp_shards}] 加载临时分片...")
    temp_data = load_file(temp_file)
    final_state_dict.update(temp_data)
    del temp_data
    force_cleanup()

    if i % 5 == 0:
        print_mem()

# 加载 scales
scales_file = os.path.join(TEMP_DIR, "temp_scales.safetensors")
if os.path.exists(scales_file):
    print(f"   加载 scales...")
    scales_data = load_file(scales_file)
    final_state_dict.update(scales_data)
    del scales_data
    force_cleanup()

print(f"\n✅ 合并统计:")
print(f"   总参数数: {len(final_state_dict)}")

# 验证
scale_keys = [k for k in final_state_dict.keys() if k.endswith('.scale')]
weight_keys = [k for k in final_state_dict.keys() if k.endswith('.weight')]

print(f"   权重参数: {len(weight_keys)}")
print(f"   Scale 参数: {len(scale_keys)}")

# 🔧 验证 scale 的 shape
print(f"\n   验证 scale shape（前 3 个）:")
for k in sorted(scale_keys)[:3]:
    scale = final_state_dict[k]
    print(f"      {k}: {list(scale.shape)}")

# 保存单文件
print(f"\n💾 保存最终文件...")

save_file(
    final_state_dict,
    os.path.join(LOCAL_DIR, "diffusion_pytorch_model.safetensors"),
    metadata={"format": "pt"}
)

file_size = os.path.getsize(os.path.join(LOCAL_DIR, "diffusion_pytorch_model.safetensors"))
print(f"   ✓ 文件大小: {file_size/1e9:.2f} GB")

del final_state_dict
force_cleanup()
print_mem()

# ==========================================
# 4. 生成配置文件
# ==========================================
print(f"\n📝 步骤 4: 生成配置文件...")

quantization_config_standalone = {
    "add_skip_keys": False,
    "dequantize_fp32": False,
    "group_size": -1,
    "is_integer": True,
    "is_training": False,
    "modules_dtype_dict": {},
    "modules_to_not_convert": target_modules_to_not_convert,
    "non_blocking": False,
    "quant_conv": False,
    "quant_method": "sdnq",
    "quantization_device": None,
    "quantized_matmul_dtype": None,
    "return_device": None,
    "sdnq_version": "0.1.2",
    "svd_rank": 32,
    "svd_steps": 8,
    "use_grad_ckpt": True,
    "use_quantized_matmul": False,
    "use_quantized_matmul_conv": False,
    "use_static_quantization": True,
    "use_stochastic_rounding": False,
    "use_svd": False,
    "weights_dtype": "int8"
}

with open(os.path.join(LOCAL_DIR, "quantization_config.json"), "w") as f:
    json.dump(quantization_config_standalone, f, indent=2)

quantization_config_embedded = {
    **quantization_config_standalone,
    "quantization_device": "xpu",
    "return_device": "cpu",
    "add_skip_keys": True,
}

base_config["quantization_config"] = quantization_config_embedded
base_config["_diffusers_version"] = "0.36.0.dev0"
base_config.pop("siglip_feat_dim", None)

with open(os.path.join(LOCAL_DIR, "config.json"), "w") as f:
    json.dump(base_config, f, indent=2)

print("   ✓ 配置文件已生成")

# ==========================================
# 5. 拷贝到 Drive
# ==========================================
print(f"\n📤 步骤 5: 拷贝到 Drive...")
print(f"   目标: {FINAL_DIR}")

if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)

shutil.copytree(LOCAL_DIR, FINAL_DIR)
print("   ✓ 拷贝完成")

print(f"\n🧹 清理临时文件...")
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
shutil.rmtree(TEMP_DIR, ignore_errors=True)
force_cleanup()

# ==========================================
# 6. 最终验证
# ==========================================
print(f"\n{'='*60}")
print(f"🎉 Per-Channel 量化完成！")
print(f"{'='*60}")

print(f"\n📁 最终文件:")
for f in sorted(os.listdir(FINAL_DIR)):
    fpath = os.path.join(FINAL_DIR, f)
    if os.path.isfile(fpath):
        fsize = os.path.getsize(fpath)
        if fsize > 1e6:
            print(f"   ✓ {f:<45} {fsize/1e9:>8.2f} GB")
        else:
            print(f"   ✓ {f:<45} {fsize/1e3:>8.1f} KB")

print(f"\n✅ 与 Turbo 对齐检查:")
print(f"   ✓ 量化方式: Per-Channel")
print(f"   ✓ Scale shape: [out_features, 1]")
print(f"   ✓ Scale dtype: bfloat16")
print(f"   ✓ 未量化层: bfloat16")
print(f"   ✓ 量化参数数: {quantized_count}")

print(f"\n🚀 使用方法:")
print(f"   !cp -rf {FINAL_DIR}/* \\")
print(f"           /content/Z-Image-Turbo-SDNQ-int8/transformer/")

print_mem()

from IPython.display import clear_output

final_message = "量化模型已保存于google driver，今后无需再运行，请重启系统，连接GPU，跳过本单元格，重新运行程序"

clear_output(wait=True)
print("✅ " + final_message)

In [ ]:
# @title 控制显存碎片配置
import os, sys

# 必须在 import torch / diffusers / transformers 之前运行
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# 防呆：如果 torch 已经被导入了，就说明这格跑晚了（本次不会生效）
if "torch" in sys.modules:
    raise RuntimeError("torch 已导入：allocator 配置本次不会生效。请重启 runtime 后第一格先跑这个 cell。")

print("✅ PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

In [ ]:
# @title 环境配置
%%capture
!pip install -U --prefer-binary -q \
  git+https://github.com/Disty0/sdnq \
  git+https://github.com/huggingface/diffusers

!pip install -U -q transformers accelerate

In [ ]:
# @title 🔥 安装 cache-dit（含量化与缓存加速支持）
%%capture
!pip install -U -q "cache-dit[quantization]"

🚀 预加载base-transformer-int8要从GDrive读取，比较慢，保持耐心！

In [ ]:
# @title 🤖 模型管理器（五合一）
import ipywidgets as widgets
from IPython.display import display
import torch
import os
import gc

# ----------------------------
# ✅ TF32 精度控制
# ----------------------------
try:
    torch.set_float32_matmul_precision('high')
    torch.backends.cudnn.conv.fp32_precision = 'tf32'
except AttributeError:
    pass

# ----------------------------
# 路径常量
# ----------------------------
PATH_INT8      = "/content/Z-Image-Turbo-SDNQ-int8"
PATH_UINT4     = "/content/Z-Image-Turbo-SDNQ-uint4"
PATH_BASE_INT8 = "/content/drive/MyDrive/z-image-base-transformer-int8"  # Google Drive 直读

# ----------------------------
# 所有文件清单（用于下载/检查）
# ----------------------------
FILES_COMMON =[
    "model_index.json",
    "scheduler/scheduler_config.json",
    "tokenizer/merges.txt",
    "tokenizer/tokenizer.json",
    "tokenizer/tokenizer_config.json",
    "tokenizer/vocab.json",
    "vae/config.json",
    "vae/diffusion_pytorch_model.safetensors",
]
FILES_TE =[
    "text_encoder/config.json",
    "text_encoder/generation_config.json",
    "text_encoder/model.safetensors",
    "text_encoder/quantization_config.json",
]
FILES_TRANSFORMER_INT8 =[
    "transformer/config.json",
    "transformer/diffusion_pytorch_model.safetensors",
    "transformer/quantization_config.json",
]
FILES_TRANSFORMER_UINT4 =[
    "transformer/config.json",
    "transformer/diffusion_pytorch_model.safetensors",
    "transformer/quantization_config.json",
]

# ----------------------------
# 五种模型配置
# ----------------------------
MODEL_CATALOG = {
    "⚡ uint4（预览/更快）": {
        "main_path"  : PATH_UINT4,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32",
        "main_files" : FILES_COMMON + FILES_TE + FILES_TRANSFORMER_UINT4,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : None,
        "transformer_is_gdrive" : False,
        "description": "完整 uint4 套件，速度最快，适合预览出图",
    },
    "🎯 int8（产品/更好）": {
        "main_path"  : PATH_INT8,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-int8",
        "main_files" : FILES_COMMON + FILES_TE + FILES_TRANSFORMER_INT8,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : None,
        "transformer_is_gdrive" : False,
        "description": "完整 int8 套件，画质最佳，适合产品出图",
    },
    "🔀 混搭①：uint4-TE + int8 主体": {
        "main_path"  : PATH_INT8,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-int8",
        "main_files" : FILES_COMMON + FILES_TRANSFORMER_INT8,
        "te_path"    : PATH_UINT4,
        "te_repo"    : "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32",
        "te_files"   : FILES_TE,
        "transformer_path"      : None,
        "transformer_is_gdrive" : False,
        "description": "int8 骨架 + uint4 TE，兼顾速度与画质",
    },
    "🔩 混搭②：uint4 + base-int8 Transformer（GDrive）": {
        "main_path"  : PATH_UINT4,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32",
        "main_files" : FILES_COMMON + FILES_TE,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : PATH_BASE_INT8,
        "transformer_is_gdrive" : True,
        "description": "uint4 骨架 + Google Drive base-int8 Transformer，直读 GDrive 无需下载",
    },
    "🔧 混搭③：int8 + base-int8 Transformer（GDrive）": {
        "main_path"  : PATH_INT8,
        "main_repo"  : "Disty0/Z-Image-Turbo-SDNQ-int8",
        "main_files" : FILES_COMMON + FILES_TE,
        "te_path"    : None,
        "te_repo"    : None,
        "te_files"   :[],
        "transformer_path"      : PATH_BASE_INT8,
        "transformer_is_gdrive" : True,
        "description": "int8 骨架 + Google Drive base-int8 Transformer，直读 GDrive 无需下载",
    },
}

# ----------------------------
# UI 组件
# ----------------------------
output = widgets.Output()

model_dropdown = widgets.Dropdown(
    options=list(MODEL_CATALOG.keys()),
    value="⚡ uint4（预览/更快）",
    description="选择方案:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%"),
)

desc_html   = widgets.HTML(value="")
repo_html   = widgets.HTML(value="")
status_html = widgets.HTML(value="<span style='color:gray;'>准备就绪</span>")
progress_html = widgets.HTML(value="")
local_status_html = widgets.HTML(value="")

path_input = widgets.Text(
    value=MODEL_CATALOG[model_dropdown.value]["main_path"],
    placeholder="主体本地路径",
    description="主体路径:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%")
)

vae_path_input = widgets.Text(
    value="",
    placeholder="（可选）留空=模型自带 VAE",
    description="VAE 路径:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%")
)
vae_status_html = widgets.HTML(value="<span style='color:gray;'>VAE：默认（使用模型自带）</span>")

mode_dropdown = widgets.Dropdown(
    options=[
        ("📁 从本地路径加载", "local"),
        ("⬇️ 下载/补全所需文件", "download"),
        ("🔧 检查并修复缺失文件", "repair"),
    ],
    value="local",
    description="操作模式:",
    style={'description_width': '70px'},
    layout=widgets.Layout(width="98%")
)

execute_btn = widgets.Button(
    description="🚀 执行",
    button_style="primary",
    layout=widgets.Layout(width="120px", height="40px")
)
cleanup_btn = widgets.Button(
    description="🧹 清除旧模型",
    button_style="warning",
    layout=widgets.Layout(width="145px", height="40px")
)

def cfg(): return MODEL_CATALOG[model_dropdown.value]
def is_gdrive_transformer(): return cfg().get("transformer_is_gdrive", False)
def has_external_te(): return cfg().get("te_path") is not None

def update_info():
    c = cfg()
    desc_html.value = f"""
    <div style='background:#1e293b;color:#e2e8f0;padding:8px 12px;border-radius:6px;font-size:12px;margin-top:4px;'>
        <b style='color:#38bdf8;'>💡 {model_dropdown.value}</b><br>
        {c['description']}
    </div>"""

    lines =[f"<b>主体 Repo:</b> <code>{c['main_repo']}</code>"]
    if c.get("te_path"):
        lines.append(f"<b>TE Repo:</b> <code>{c['te_repo']}</code> → <code>{c['te_path']}/text_encoder/</code>")
    if c.get("transformer_path"):
        src = "☁️ Google Drive（直读，不下载）" if c["transformer_is_gdrive"] else c["transformer_path"]
        lines.append(f"<b>Transformer:</b> {src}")
    repo_html.value = "<div style='font-size:12px;color:#64748b;margin:4px 0;'>" + "<br>".join(lines) + "</div>"

def refresh_local_status():
    c = cfg()
    main = path_input.value.strip()
    msgs = []
    missing_main =[f for f in c["main_files"] if not _file_ok(main, f)]
    if missing_main: msgs.append(f"主体缺失 {len(missing_main)} 个文件")
    if c.get("te_path"):
        missing_te = [f for f in c["te_files"] if not _file_ok(c["te_path"], f)]
        if missing_te: msgs.append(f"TE 缺失 {len(missing_te)} 个文件")
    if c.get("transformer_path") and c["transformer_is_gdrive"]:
        tp = c["transformer_path"]
        tw = os.path.join(tp, "diffusion_pytorch_model.safetensors")
        if not os.path.exists(tw): msgs.append(f"⚠️ GDrive transformer 未找到: {tp}")

    if msgs: local_status_html.value = f"<span style='color:orange;'>⚠️ {' | '.join(msgs)}</span>"
    else: local_status_html.value = "<span style='color:green;'>✅ 所需文件完整，可直接加载</span>"

def _file_ok(base, rel):
    p = os.path.join(base, rel)
    return os.path.exists(p) and os.path.getsize(p) > 0

def refresh_vae_status():
    p = (vae_path_input.value or "").strip()
    if not p:
        vae_status_html.value = "<span style='color:gray;'>VAE：默认（使用模型自带）</span>"
        return
    try:
        vd = _resolve_vae_dir(p)
        vae_status_html.value = (f"<span style='color:green;'>✅ 外置 VAE: {vd}</span>")
    except Exception as e:
        vae_status_html.value = f"<span style='color:red;'>❌ VAE 路径无效：{e}</span>"

def on_model_change(change):
    path_input.value = cfg()["main_path"]
    update_info()
    refresh_local_status()

model_dropdown.observe(on_model_change, names="value")
path_input.observe(lambda c: refresh_local_status(), names="value")
vae_path_input.observe(lambda c: refresh_vae_status(), names="value")
update_info()
refresh_local_status()
refresh_vae_status()

# ----------------------------
# 下载 / 修复
# ----------------------------
def download_file(repo_id, filename, save_path):
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(
            repo_id=repo_id, filename=filename,
            local_dir=save_path, local_dir_use_symlinks=False, force_download=True,
        )
        lf = os.path.join(save_path, filename)
        if os.path.exists(lf) and os.path.getsize(lf) > 0: return True, os.path.getsize(lf)
        return False, 0
    except Exception as e:
        return False, str(e)

def download_filelist(repo_id, base_path, file_list, label=""):
    os.makedirs(base_path, exist_ok=True)
    results = {"success": 0, "failed": 0, "skipped": 0}
    failed =[]
    for i, f in enumerate(file_list):
        progress_html.value = f"<span style='color:blue;'>📥 [{label}{i+1}/{len(file_list)}] {f}</span>"
        if _file_ok(base_path, f):
            sz = os.path.getsize(os.path.join(base_path, f)) / 1024**2
            print(f"⏭️ 已存在：{f} ({sz:.1f} MB)")
            results["skipped"] += 1
            continue
        print(f"⬇️ 下载：{f}...")
        ok, info = download_file(repo_id, f, base_path)
        if ok:
            print(f"   ✅ {info/1024**2:.1f} MB")
            results["success"] += 1
        else:
            print(f"   ❌ {info}")
            results["failed"] += 1
            failed.append(f)
    progress_html.value = ""
    return results, failed

def do_download():
    c = cfg()
    main_path = path_input.value.strip()
    total = {"success":0, "failed":0, "skipped":0}
    all_failed = []

    print(f"📦 [主体] {c['main_repo']} → {main_path}")
    print(f"   文件数：{len(c['main_files'])}")
    print("-" * 55)
    r, f = download_filelist(c["main_repo"], main_path, c["main_files"], "主体 ")
    for k in total: total[k] += r[k]
    all_failed += f

    if c.get("te_path") and c.get("te_repo"):
        print(f"\n📦 [TE] {c['te_repo']} → {c['te_path']}")
        print(f"   文件数：{len(c['te_files'])}")
        print("-" * 55)
        r2, f2 = download_filelist(c["te_repo"], c["te_path"], c["te_files"], "TE ")
        for k in total: total[k] += r2[k]
        all_failed += f2

    if c.get("transformer_path") and c["transformer_is_gdrive"]:
        tp = c["transformer_path"]
        tw = os.path.join(tp, "diffusion_pytorch_model.safetensors")
        print(f"\n☁️[GDrive Transformer] 路径：{tp}")
        if os.path.exists(tw): print(f"   ✅ safetensors 已存在，直读即用")
        else: print(f"   ⚠️  文件不存在！请确认 Google Drive 已挂载且路径正确。")

    print("=" * 55)
    print(f"📊 下载 {total['success']}  跳过 {total['skipped']}  失败 {total['failed']}")
    if all_failed:
        print("❌ 失败文件：")
        for f in all_failed: print(f"   - {f}")
    return total, all_failed

def do_repair():
    c = cfg()
    main_path = path_input.value.strip()
    total_fixed, total_needed = 0, 0
    any_fail = False

    missing_main = [f for f in c["main_files"] if not _file_ok(main_path, f)]
    if missing_main:
        print(f"🔧 [主体] 缺失 {len(missing_main)} 文件，开始修复...")
        os.makedirs(main_path, exist_ok=True)
        total_needed += len(missing_main)
        for i, f in enumerate(missing_main):
            progress_html.value = f"<span style='color:orange;'>🔧 [主体 {i+1}/{len(missing_main)}] {f}</span>"
            print(f"⬇️ {f}...")
            ok, info = download_file(c["main_repo"], f, main_path)
            if ok:
                print(f"   ✅ {info/1024**2:.1f} MB"); total_fixed += 1
            else:
                print(f"   ❌ {info}"); any_fail = True
    else:
        print("✅ 主体文件完整")

    if c.get("te_path") and c.get("te_repo"):
        missing_te = [f for f in c["te_files"] if not _file_ok(c["te_path"], f)]
        if missing_te:
            print(f"\n🔧 [TE] 缺失 {len(missing_te)} 文件，开始修复...")
            os.makedirs(c["te_path"], exist_ok=True)
            total_needed += len(missing_te)
            for i, f in enumerate(missing_te):
                progress_html.value = f"<span style='color:orange;'>🔧 [TE {i+1}/{len(missing_te)}] {f}</span>"
                print(f"⬇️ {f}...")
                ok, info = download_file(c["te_repo"], f, c["te_path"])
                if ok:
                    print(f"   ✅ {info/1024**2:.1f} MB"); total_fixed += 1
                else:
                    print(f"   ❌ {info}"); any_fail = True
        else:
            print("✅ TE 文件完整")

    if c.get("transformer_path") and c["transformer_is_gdrive"]:
        tp = c["transformer_path"]
        tw = os.path.join(tp, "diffusion_pytorch_model.safetensors")
        if os.path.exists(tw): print(f"\n✅ GDrive Transformer 存在：{tp}")
        else:
            print(f"\n⚠️  GDrive Transformer 未找到：{tp}")
            print(f"   请确认 Google Drive 已挂载且路径正确（该文件不会自动下载）")
            any_fail = True

    progress_html.value = ""
    print("-" * 40)
    print(f"📊 修复：{total_fixed}/{total_needed}")
    return not any_fail

def _resolve_vae_dir(user_path: str) -> str:
    p = (user_path or "").strip()
    if not p: return ""
    if not os.path.exists(p): raise RuntimeError(f"VAE 路径不存在：{p}")
    if os.path.isdir(p) and os.path.exists(os.path.join(p, "config.json")): return p
    cand = os.path.join(p, "vae")
    if os.path.isdir(cand) and os.path.exists(os.path.join(cand, "config.json")): return cand
    for dirpath, _, fnames in os.walk(p):
        if "config.json" in fnames:
            if "diffusion_pytorch_model.safetensors" in fnames or "diffusion_pytorch_model.bin" in fnames:
                if os.path.basename(dirpath).lower() == "vae": return dirpath
    raise RuntimeError(f"找不到可用 VAE 目录：{p}")

def maybe_override_vae(pipe, vae_path_text, local_only_hint):
    vae_path_text = (vae_path_text or "").strip()
    if not vae_path_text: return False, "VAE：默认（使用模型自带）"
    vae_dir = _resolve_vae_dir(vae_path_text)
    from diffusers import AutoencoderKL
    vae = AutoencoderKL.from_pretrained(
        vae_dir,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        local_files_only=True,
    )
    old = getattr(pipe, "vae", None)
    pipe.vae = vae
    try:
        if torch.cuda.is_available(): pipe.vae.to("cuda")
    except Exception: pass
    try:
        if old is not None: del old
    except Exception: pass
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    return True, f"VAE：已覆盖 → {vae_dir}"

def set_pipe_model_root(pipe, main_path, te_path=None, transformer_path=None):
    pipe._model_root = main_path
    pipe._te_path = te_path
    pipe._transformer_override_path = transformer_path

def drop_text_encoder(pipe, drop_tokenizer=False):
    try:
        te = getattr(pipe, "text_encoder", None)
        pipe.text_encoder = None
        if te is not None: del te
    except Exception: pass
    if drop_tokenizer:
        try:
            tok = getattr(pipe, "tokenizer", None)
            pipe.tokenizer = None
            if tok is not None: del tok
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def ensure_text_encoder_loaded(pipe, device="cpu", dtype=torch.float32):
    if getattr(pipe, "text_encoder", None) is not None: return pipe.text_encoder
    model_root = getattr(pipe, "_model_root", None)
    te_path    = getattr(pipe, "_te_path", None)
    if model_root is None: raise RuntimeError("pipe._model_root 未设置")

    is_hybrid = te_path is not None and te_path != model_root
    if is_hybrid:
        te_dir = os.path.join(te_path, "text_encoder")
        if not os.path.exists(te_dir): raise RuntimeError(f"找不到外置 TE 目录：{te_dir}")
        print(f"🔀 混合 TE：使用 {te_dir}")
    else:
        te_dir = os.path.join(model_root, "text_encoder")
        if not os.path.exists(te_dir): raise RuntimeError(f"找不到 text_encoder 目录：{te_dir}")

    from transformers import AutoModel
    from sdnq.loader import apply_sdnq_options_to_model
    te = AutoModel.from_pretrained(te_dir, local_files_only=True)
    te = apply_sdnq_options_to_model(te, use_quantized_matmul=True)
    try: te.to(device)
    except Exception: pass
    pipe.text_encoder = te
    return te

def ensure_tokenizer_loaded(pipe):
    if getattr(pipe, "tokenizer", None) is not None: return pipe.tokenizer
    root = getattr(pipe, "_model_root", None)
    if root is None: raise RuntimeError("pipe._model_root 未设置")
    tok_dir = os.path.join(root, "tokenizer")
    if not os.path.exists(tok_dir): raise RuntimeError(f"找不到 tokenizer 目录：{tok_dir}")
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(tok_dir, local_files_only=True)
    pipe.tokenizer = tok
    return tok

import functools
class ZImageDiTCache:
    def __init__(self, pipe, residual_diff_threshold=0.08, warmup_steps=2, skip_last_n_blocks=2):
        self.pipe = pipe
        self.threshold = residual_diff_threshold
        self.warmup_steps = warmup_steps
        self.skip_last_n = skip_last_n_blocks
        self._patched = False
        self._original_forwards = {}
        self._global_step = 0
        self._total_blocks = 0
        self._skipped_blocks = 0

    def enable(self):
        if self._patched:
            print("⚠️ 缓存已启用，跳过重复初始化")
            return
        transformer = self.pipe.transformer
        blocks = transformer.layers
        n = len(blocks)
        for i, block in enumerate(blocks):
            self._patch_block(block, idx=i, cache_enabled=(i < n - self.skip_last_n))
        self._register_step_hook()
        self._patched = True
        cached_count = n - self.skip_last_n
        print(f"✅ ZImageDiTCache 启用：缓存 layers[0..{cached_count-1}]，跳过最后 {self.skip_last_n} 块")

    def _patch_block(self, block, idx, cache_enabled):
        original_fwd = block.forward
        self._original_forwards[idx] = original_fwd
        cache_state = {"cached_output": None, "last_hidden_states": None, "hit_count": 0, "miss_count": 0}
        diT_cache = self

        @functools.wraps(original_fwd)
        def cached_forward(hidden_states, *args, **kwargs):
            threshold = diT_cache.threshold
            warmup = diT_cache.warmup_steps
            if diT_cache._global_step < warmup or not cache_enabled:
                out = original_fwd(hidden_states, *args, **kwargs)
                cache_state["cached_output"] = out
                cache_state["last_hidden_states"] = hidden_states.detach()
                cache_state["miss_count"] += 1
                return out
            if cache_state["last_hidden_states"] is not None:
                with torch.no_grad():
                    diff = (hidden_states.float() - cache_state["last_hidden_states"].float()).abs().mean().item()
                if diff < threshold and cache_state["cached_output"] is not None:
                    cache_state["hit_count"] += 1
                    diT_cache._skipped_blocks += 1
                    return cache_state["cached_output"]
            out = original_fwd(hidden_states, *args, **kwargs)
            cache_state["cached_output"] = out
            cache_state["last_hidden_states"] = hidden_states.detach()
            cache_state["miss_count"] += 1
            return out

        block.forward = cached_forward
        block._cache_state = cache_state

    def _register_step_hook(self):
        pipe = self.pipe
        diT_cache = self
        original_step = pipe.scheduler.step

        @functools.wraps(original_step)
        def step_with_counter(*args, **kwargs):
            diT_cache._global_step += 1
            diT_cache._total_blocks += len(pipe.transformer.layers)
            return original_step(*args, **kwargs)

        pipe.scheduler.step = step_with_counter

    def reset(self):
        self._global_step = 0
        self._total_blocks = 0
        self._skipped_blocks = 0
        for block in self.pipe.transformer.layers:
            if hasattr(block, '_cache_state'):
                block._cache_state["last_hidden_states"] = None
                block._cache_state["cached_output"] = None

    def stats(self):
        if self._total_blocks == 0:
            print("📊 暂无统计（尚未推理）")
            return
        rate = self._skipped_blocks / self._total_blocks * 100
        print(f"📊 缓存：总 {self._total_blocks}  跳过 {self._skipped_blocks}（{rate:.1f}%）  实算 {self._total_blocks - self._skipped_blocks}")

    def disable(self):
        if not self._patched: return
        for i, block in enumerate(self.pipe.transformer.layers):
            if i in self._original_forwards:
                block.forward = self._original_forwards[i]
        self._patched = False
        print("✅ 缓存已禁用")

_dit_cache_instance = None

def try_enable_cache_dit(pipe):
    global _dit_cache_instance
    try:
        if pipe.transformer is None:
            print("❌ transformer 为 None，跳过缓存")
            return False
        _dit_cache_instance = ZImageDiTCache(pipe, residual_diff_threshold=0.08, warmup_steps=2, skip_last_n_blocks=2)
        _dit_cache_instance.enable()
        return True
    except Exception as e:
        print(f"⚠️ ZImageDiTCache 失败: {e}")
        import traceback; traceback.print_exc()
        return False

# ----------------------------
# 🚀 加载模型（核心）
# ----------------------------
def _load_transformer_from_path(transformer_path, transformer_is_gdrive):
    import diffusers, json
    gdrive_note = "（Google Drive 直读）" if transformer_is_gdrive else ""
    print(f"🔩 预加载 Transformer：{transformer_path} {gdrive_note}")

    config_path = os.path.join(transformer_path, "config.json")
    if not os.path.exists(config_path): raise RuntimeError(f"找不到 Transformer config.json：{config_path}")

    with open(config_path) as f: trans_cfg = json.load(f)
    class_name = trans_cfg.get("_class_name", "")

    candidate_classes =[]
    if class_name and hasattr(diffusers, class_name): candidate_classes.append(getattr(diffusers, class_name))
    for name in ["ZImageTransformer2DModel", "Transformer2DModel", "DiTTransformer2DModel"]:
        cls = getattr(diffusers, name, None)
        if cls and cls not in candidate_classes: candidate_classes.append(cls)

    last_err = None
    for cls in candidate_classes:
        try:
            t = cls.from_pretrained(transformer_path, torch_dtype=torch.float32, local_files_only=True)
            print(f"   ✅ 加载成功：{cls.__name__}")
            return t
        except Exception as e:
            last_err = e
            continue

    try:
        from diffusers import AutoModel as DiffusersAutoModel
        t = DiffusersAutoModel.from_pretrained(transformer_path, torch_dtype=torch.float32, local_files_only=True)
        print(f"   ✅ AutoModel fallback 成功：{type(t).__name__}")
        return t
    except Exception as e2:
        raise RuntimeError(f"无法加载 Transformer\n最后错误：{last_err}\nAutoModel 错误：{e2}")

def load_model(main_path, te_path=None, transformer_path=None, transformer_is_gdrive=False):
    global pipe
    import diffusers
    from sdnq.loader import apply_sdnq_options_to_model

    load_kwargs = {"torch_dtype": torch.float32, "device_map": "cuda", "text_encoder": None, "local_files_only": True}

    if transformer_path:
        pre_transformer = _load_transformer_from_path(transformer_path, transformer_is_gdrive)
        load_kwargs["transformer"] = pre_transformer
        print(f"   ☑️  Transformer 已预加载，pipeline 将跳过主体 transformer 目录")

    pipe = diffusers.ZImagePipeline.from_pretrained(main_path, **load_kwargs)
    set_pipe_model_root(pipe, main_path, te_path=te_path, transformer_path=transformer_path)

    if transformer_path:
        try:
            if torch.cuda.is_available(): pipe.transformer.to("cuda")
        except Exception: pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"   ✅ Transformer 就位：{type(pipe.transformer).__name__}")

    pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
    drop_text_encoder(pipe, drop_tokenizer=False)

    assert pipe.transformer is not None, "transformer 加载异常"
    print(f"✅ transformer 就绪：{type(pipe.transformer).__name__}")
    try_enable_cache_dit(pipe)
    return pipe

# ----------------------------
# 清除与执行操作
# ----------------------------
def hard_cleanup():
    global pipe
    freed =[]
    if "pipe" in globals() and pipe is not None:
        try: del pipe; freed.append("pipe")
        except Exception: pass
        finally: pipe = None
    if "loaded_components" in globals():
        try: del globals()["loaded_components"]; freed.append("loaded_components")
        except Exception: pass
    if torch.cuda.is_available():
        try: torch.cuda.empty_cache()
        except Exception: pass
        try: torch.cuda.ipc_collect()
        except Exception: pass
    try: gc.collect()
    except Exception: pass
    return freed

def on_cleanup_click(b):
    output.clear_output()
    cleanup_btn.disabled = True
    with output:
        try:
            status_html.value = "<span style='color:orange;'>🧹 清理中...</span>"
            freed = hard_cleanup()
            print("🧹 清理完成\n释放：", ", ".join(freed) if freed else "（无可释放对象）")
            if torch.cuda.is_available():
                a = torch.cuda.memory_allocated() / 1024**3
                r = torch.cuda.memory_reserved() / 1024**3
                print(f"💾 显存：{a:.2f} GB (使用) / {r:.2f} GB (保留)")
            status_html.value = "<span style='color:green;'>✅ 已清理</span>"
        except Exception as e:
            print("❌ 清理失败:", e)
            status_html.value = "<span style='color:red;'>❌ 清理失败</span>"
        finally:
            cleanup_btn.disabled = False

cleanup_btn.on_click(on_cleanup_click)

def on_execute_click(b):
    global pipe
    output.clear_output()
    execute_btn.disabled = True
    mode = mode_dropdown.value
    c = cfg()
    main_path   = path_input.value.strip()
    vae_path    = vae_path_input.value.strip()
    te_path     = c.get("te_path")
    trans_path  = c.get("transformer_path")
    trans_gdrive= c.get("transformer_is_gdrive", False)

    with output:
        try:
            if mode == "download":
                print(f"⬇️ 下载模式：{model_dropdown.value}\n" + "="*55)
                status_html.value = "<span style='color:orange;'>⏳ 下载中...</span>"
                results, failed = do_download()
                if results["failed"] == 0:
                    print("\n✅ 下载完成！")
                    status_html.value = "<span style='color:green;'>✅ 下载完成</span>"
                else: status_html.value = f"<span style='color:orange;'>⚠️ {results['failed']} 个文件失败</span>"
            elif mode == "repair":
                print(f"🔧 修复模式：{model_dropdown.value}\n" + "="*55)
                status_html.value = "<span style='color:orange;'>⏳ 检查中...</span>"
                ok = do_repair()
                status_html.value = "<span style='color:green;'>✅ 已完整</span>" if ok else "<span style='color:red;'>❌ 仍有文件缺失</span>"
            elif mode == "local":
                print(f"📁 本地加载：{model_dropdown.value}")
                print(f"   主体路径：{main_path}")
                if te_path: print(f"   外置 TE：{te_path}/text_encoder/")
                if trans_path: print(f"   Transformer：{trans_path} " + ("（GDrive 直读）" if trans_gdrive else ""))
                print("-" * 55)

                missing_main = [f for f in c["main_files"] if not _file_ok(main_path, f)]
                missing_te   =[f for f in c.get("te_files", []) if not _file_ok(te_path or "", f)] if te_path else[]
                if missing_main or missing_te:
                    print("❌ 文件不完整！请切换到下载/修复模式")
                    status_html.value = "<span style='color:red;'>❌ 文件不完整</span>"
                    return
                if trans_path:
                    tw = os.path.join(trans_path, "diffusion_pytorch_model.safetensors")
                    if not os.path.exists(tw):
                        print(f"❌ Transformer 权重不存在：{tw}")
                        status_html.value = "<span style='color:red;'>❌ Transformer 路径无效</span>"
                        return

                status_html.value = "<span style='color:orange;'>⏳ 加载中...</span>"
                if "pipe" in globals() and pipe is not None:
                    del pipe
                    if torch.cuda.is_available(): torch.cuda.empty_cache()
                    gc.collect()

                pipe = load_model(main_path=main_path, te_path=te_path, transformer_path=trans_path, transformer_is_gdrive=trans_gdrive)
                _, vae_msg = maybe_override_vae(pipe, vae_path, local_only_hint=True)
                print("\n✅ 模型加载完成！")
                print(f"📌 方案：{model_dropdown.value}\n🧩 {vae_msg}")
                status_html.value = f"<span style='color:green;'>✅ 已加载</span>"

            if torch.cuda.is_available() and "pipe" in globals() and pipe is not None:
                print(f"\n💾 显存：{torch.cuda.memory_allocated() / 1024**3:.2f} GB / {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

        except Exception as e:
            print(f"\n❌ 错误：{e}")
            status_html.value = "<span style='color:red;'>❌ 失败</span>"
            import traceback; traceback.print_exc()
        finally:
            execute_btn.disabled = False
            progress_html.value = ""
            refresh_local_status()
            refresh_vae_status()

execute_btn.on_click(on_execute_click)

# ----------------------------
# UI 布局
# ----------------------------
header = widgets.HTML(value="""
<div style='background:linear-gradient(135deg,#0f2027 0%,#203a43 50%,#2c5364 100%);
            color:white; padding:12px 16px; border-radius:8px; margin-bottom:8px;'>
    <h3 style='margin:0; color:#38bdf8;'>🤖 模型管理器（五合一）</h3>
    <span style='font-size:12px; color:#94a3b8;'>
        uint4 / int8 / 混搭①(uint4-TE+int8) / 混搭②(uint4+GDrive-Transformer) / 混搭③(int8+GDrive-Transformer)
    </span>
</div>
""")

layout = widgets.VBox([
    header, model_dropdown, desc_html, repo_html, local_status_html,
    widgets.HTML("<hr style='margin:8px 0; border-color:#334155;'>"),
    mode_dropdown, path_input, vae_path_input, vae_status_html,
    widgets.HBox([execute_btn, cleanup_btn, status_html], layout=widgets.Layout(gap="10px", align_items="center")),
    progress_html, output
], layout=widgets.Layout(padding="12px", border="1px solid #334155", border_radius="10px", width="760px"))

display(layout)

🚀 Distill融合后，出图时CFG=0！，steps按照文件名标识x2

In [ ]:
# @title 🧩 Lora-Distill 下载 / 转换 / 融合 / 卸载
import ipywidgets as widgets
from IPython.display import display
import torch, os, gc, re, requests
from pathlib import Path
import safetensors.torch
from huggingface_hub import hf_hub_download, list_repo_files

# ════════════════════════════════════════════════════════
# 1. 转换映射（第一段 ATTENTION_MAP + 第二段修正 FF_MAP）
# ════════════════════════════════════════════════════════
ATTENTION_MAP = {
    "attention_to_q":     "attention.to_q",
    "attention_to_k":     "attention.to_k",
    "attention_to_v":     "attention.to_v",
    "attention_to_out_0": "attention.to_out.0",
    "attention_to_out":   "attention.to_out.0",
}
FF_MAP = {
    "feed_forward_w1": "feed_forward.w1",
    "feed_forward_w2": "feed_forward.w2",
    "feed_forward_w3": "feed_forward.w3",
}
SUB_MAP    = {**ATTENTION_MAP, **FF_MAP}
SUFFIX_MAP = {
    "lora_down.weight": "lora_A.weight",
    "lora_up.weight":   "lora_B.weight",
}
PATTERN = re.compile(
    r"^lora_unet__(layers|context_refiner|noise_refiner)"
    r"_(\d+)_(attention_to_\w+|feed_forward_w\d)"
    r"\.(lora_down\.weight|lora_up\.weight|alpha)$"
)

def convert_key(old_key):
    m = PATTERN.match(old_key)
    if not m:
        return None, "skip"
    block_type, idx, sub, suffix = m.groups()
    if suffix == "alpha":
        return None, "alpha"
    mapped_sub    = SUB_MAP.get(sub)
    mapped_suffix = SUFFIX_MAP.get(suffix)
    if not mapped_sub or not mapped_suffix:
        return None, "skip"
    return f"diffusion_model.{block_type}.{idx}.{mapped_sub}.{mapped_suffix}", "ok"

def needs_conversion(file_path):
    try:
        tensors = safetensors.torch.load_file(file_path, device="cpu")
        return any(PATTERN.match(k) for k in tensors.keys())
    except Exception:
        return False

# ════════════════════════════════════════════════════════
# 2. 下载工具（来自第一段）
# ════════════════════════════════════════════════════════
LORA_DIR = "/content/Lora"

def download_from_url(url, custom_filename=None, force=False):
    os.makedirs(LORA_DIR, exist_ok=True)
    filename = custom_filename or os.path.basename(url.split("?")[0])
    if not filename.endswith(".safetensors"):
        filename += ".safetensors"
    local_path = os.path.join(LORA_DIR, filename)
    if os.path.exists(local_path) and not force:
        return local_path, "already_exists"
    try:
        r = requests.get(url, stream=True, timeout=30)
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return local_path, "downloaded"
    except Exception as e:
        return None, str(e)

def download_from_hub(repo_id, filename, force=False):
    os.makedirs(LORA_DIR, exist_ok=True)
    local_path = os.path.join(LORA_DIR, os.path.basename(filename))
    if os.path.exists(local_path) and not force:
        return local_path, "already_exists"
    try:
        downloaded = hf_hub_download(
            repo_id=repo_id, filename=filename,
            local_dir=LORA_DIR, local_dir_use_symlinks=False,
            force_download=force,
        )
        return downloaded, "downloaded"
    except Exception as e:
        return None, str(e)

def list_repo_safetensors(repo_id):
    try:
        return [f for f in list_repo_files(repo_id) if f.endswith(".safetensors")]
    except Exception:
        return []

# ════════════════════════════════════════════════════════
# 3. 转换函数
# ════════════════════════════════════════════════════════
def convert_lora_file(src_path, delete_original=False, log=print):
    dst_path = str(Path(src_path).parent / (Path(src_path).stem + "_diffusers.safetensors"))
    if os.path.exists(dst_path):
        os.remove(dst_path)
        log(f"🗑️ 已删除旧转换文件: {Path(dst_path).name}")
    try:
        sd = safetensors.torch.load_file(src_path, device="cpu")
        converted, dropped, skipped = {}, [], []
        for k, v in sd.items():
            new_key, status = convert_key(k)
            if status == "ok":
                converted[new_key] = v
            elif status == "alpha":
                dropped.append(k)
            else:
                skipped.append(k)
        log(f"✅ 转换: {len(converted)}  丢弃(alpha): {len(dropped)}  跳过: {len(skipped)}")
        safetensors.torch.save_file(converted, dst_path)
        size_mb = Path(dst_path).stat().st_size / 1024 / 1024
        log(f"✅ 已保存: {Path(dst_path).name}  ({size_mb:.1f} MB)")
        if delete_original:
            os.remove(src_path)
            log(f"🗑️ 已删除原文件: {Path(src_path).name}")
        return dst_path
    except Exception as e:
        log(f"❌ 转换失败: {e}")
        return None

# ════════════════════════════════════════════════════════
# 4. 融合 / 卸载 —— forward hook 方案（兼容 INT8 量化模型）
#
#  原理：不修改 int8 权重，而是在每个线性层的 forward 输出上
#  叠加 LoRA 分支：  y = W(x) + (B @ A @ x) * scale
#  卸载时只需 remove_hook，权重完全不变。
# ════════════════════════════════════════════════════════
_fused_state = {}   # 全局融合状态

def _make_lora_hook(A, B, scale, is_ff):
    """返回一个 forward hook，在线性层输出上叠加 LoRA 增量。"""
    # A: [r, in]   B: [out, r]   —— 均为 cpu float32
    # is_ff: feed_forward 层的输入是 [batch, seq, in]，输出 [batch, seq, out]
    #        attention 层同理；矩阵乘法方向一致，无需额外转置
    def hook(module, input, output):
        x = input[0].to(dtype=torch.float32)          # 升精度
        # x shape: [B, T, in_features]
        lora_out = (x @ A.T.to(x.device)) @ B.T.to(x.device)  # [B, T, out]
        lora_out = lora_out * scale
        return output + lora_out.to(dtype=output.dtype)
    return hook

def fuse_lora(lora_path, scale=1.0, log=print):
    global _fused_state
    if _fused_state.get("hooks"):
        log(f"⚠️ 检测到已注册的 LoRA hook，先自动卸载: {_fused_state['file']}")
        unfuse_lora(log=log)

    sd = safetensors.torch.load_file(lora_path, device="cpu")

    # 构建模块名 → 模块 的查找表
    module_map = dict(pipe.transformer.named_modules())

    # 收集 base_keys
    base_keys = set()
    for k in sd:
        for sfx in [".lora_A.weight", ".lora_B.weight"]:
            if k.endswith(sfx):
                base_keys.add(k[:-len(sfx)])
                break

    hooks = []
    n_attn, n_ff, n_fail = 0, 0, 0

    for bk in base_keys:
        # LoRA 键: diffusion_model.layers.0.attention.to_q
        # 模块键:  layers.0.attention.to_q
        bare = bk.removeprefix("diffusion_model.").removeprefix("transformer.")
        module = module_map.get(bare)
        if module is None:
            n_fail += 1
            continue

        A = sd[bk + ".lora_A.weight"].float()   # [r, in]
        B = sd[bk + ".lora_B.weight"].float()   # [out, r]
        is_ff = "feed_forward" in bk

        hook_fn = _make_lora_hook(A, B, scale, is_ff)
        h = module.register_forward_hook(hook_fn)
        hooks.append(h)

        if is_ff:
            n_ff += 1
        else:
            n_attn += 1

    total = n_attn + n_ff
    log(f"✅ 已注册 LoRA hook {total} 个  (attention={n_attn}, feed_forward={n_ff}, 失败={n_fail})")
    log(f"   模式: forward hook（INT8 权重不变，推理时动态叠加）")
    _fused_state = {
        "file":  Path(lora_path).name,
        "scale": scale,
        "sd":    sd,
        "hooks": hooks,
    }

def unfuse_lora(log=print):
    global _fused_state
    if not _fused_state.get("hooks"):
        log("⚠️ 当前没有已注册的 LoRA hook，无需卸载")
        return
    for h in _fused_state["hooks"]:
        h.remove()
    n = len(_fused_state["hooks"])
    log(f"✅ 已卸载 {n} 个 LoRA hook，模型权重完全还原")
    _fused_state = {}
    gc.collect()

# ════════════════════════════════════════════════════════
# 5. UI
# ════════════════════════════════════════════════════════
output_widget = widgets.Output()

# ── 下载方式 ──────────────────────────────────────────
download_method = widgets.RadioButtons(
    options=[("🔗 URL下载", "url"), ("📦 Repo ID下载", "repo")],
    value="repo",
    description="下载方式:",
    layout=widgets.Layout(width="300px")
)
url_input = widgets.Text(
    placeholder="https://huggingface.co/.../resolve/main/model.safetensors",
    layout=widgets.Layout(flex="1 1 auto")
)
url_row = widgets.HBox(
    [widgets.Label("URL:", layout=widgets.Layout(width="70px")), url_input],
    layout=widgets.Layout(width="100%", display="none")
)
repo_input = widgets.Text(
    value="alibaba-pai/Z-Image-Fun-Lora-Distill",
    layout=widgets.Layout(flex="1 1 auto")
)
browse_btn = widgets.Button(description="📂 浏览文件", layout=widgets.Layout(width="110px"))
repo_row = widgets.HBox(
    [widgets.Label("Repo ID:", layout=widgets.Layout(width="70px")), repo_input, browse_btn],
    layout=widgets.Layout(width="100%")
)
filename_input = widgets.Text(
    placeholder="文件名（例如: lora.safetensors）",
    layout=widgets.Layout(flex="1 1 auto")
)
filename_row = widgets.HBox(
    [widgets.Label("文件名:", layout=widgets.Layout(width="70px")), filename_input],
    layout=widgets.Layout(width="100%")
)
file_selector = widgets.Select(options=[], rows=6, layout=widgets.Layout(flex="1 1 auto", display="none"))
confirm_file_btn = widgets.Button(
    description="✅ 确认", button_style="success",
    layout=widgets.Layout(width="80px", display="none")
)
file_select_row = widgets.HBox(
    [widgets.Label("选择:", layout=widgets.Layout(width="70px")), file_selector, confirm_file_btn],
    layout=widgets.Layout(width="100%")
)

# ── 选项 ──────────────────────────────────────────────
force_check   = widgets.Checkbox(value=False, description="强制重新下载", indent=False)
del_orig_check = widgets.Checkbox(value=True,  description="转换后删除原文件", indent=False)

# ── Merge Scale ───────────────────────────────────────
scale_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=2.0, step=0.05,
    description="Merge Scale:",
    readout_format=".2f",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px")
)

# ── 操作模式 ──────────────────────────────────────────
mode_dropdown = widgets.Dropdown(
    options=[
        ("⬇️  下载",             "download"),
        ("🔧  转换",             "convert"),
        ("⬇️🔧 下载+转换",       "download_convert"),
        ("🔗  融合到模型",        "fuse"),
        ("⬇️🔧🔗 下载+转换+融合", "download_convert_fuse"),
        ("🔓  卸载LoRA",         "unfuse"),
    ],
    value="download_convert",
    description="操作:",
    layout=widgets.Layout(width="320px")
)

execute_btn = widgets.Button(
    description="🚀 执行", button_style="primary",
    layout=widgets.Layout(width="110px", height="38px")
)
refresh_btn = widgets.Button(
    description="🔄 刷新列表", layout=widgets.Layout(width="110px", height="38px")
)
status_html    = widgets.HTML(value="<span style='color:gray;'>准备就绪</span>")
fused_html     = widgets.HTML(value="<span style='color:gray;'>未融合</span>")
file_list_html = widgets.HTML(value="<i>暂无文件</i>")

# ── 动态 UI ───────────────────────────────────────────
def on_method_change(change):
    if change["new"] == "url":
        url_row.layout.display  = "flex"
        repo_row.layout.display = "none"
    else:
        url_row.layout.display  = "none"
        repo_row.layout.display = "flex"
    file_selector.layout.display    = "none"
    confirm_file_btn.layout.display = "none"

download_method.observe(on_method_change, names="value")

def on_browse(b):
    repo = repo_input.value.strip()
    if not repo:
        with output_widget:
            output_widget.clear_output(); print("❌ 请填写 Repo ID")
        return
    with output_widget:
        output_widget.clear_output()
        print(f"📡 正在获取 {repo} 的文件列表...")
        files = list_repo_safetensors(repo)
        if not files:
            print("❌ 未找到 .safetensors 文件或仓库不存在")
            file_selector.options = []
        else:
            file_selector.options = files
            file_selector.value   = files[0]
            file_selector.layout.display    = "flex"
            confirm_file_btn.layout.display = "flex"
            print(f"✅ 找到 {len(files)} 个文件，请选择后点击「确认」")

browse_btn.on_click(on_browse)

def on_confirm_file(b):
    sel = file_selector.value
    if sel:
        filename_input.value = sel
        file_selector.layout.display    = "none"
        confirm_file_btn.layout.display = "none"
        with output_widget:
            output_widget.clear_output(); print(f"✅ 已选择: {sel}")

confirm_file_btn.on_click(on_confirm_file)

# ── 文件列表刷新 ──────────────────────────────────────
def refresh_file_list():
    if not os.path.exists(LORA_DIR):
        file_list_html.value = "<i>目录不存在</i>"
        return
    files = [f for f in os.listdir(LORA_DIR) if f.endswith(".safetensors")]
    if not files:
        file_list_html.value = "<i>没有 .safetensors 文件</i>"
        return
    lines = []
    for f in sorted(files):
        path = os.path.join(LORA_DIR, f)
        size = os.path.getsize(path) / 1024 / 1024
        mark = "⚠️ 需转换" if needs_conversion(path) else "✅ 可用"
        lines.append(f"<div>{mark}  {f}  ({size:.2f} MB)</div>")
    state = _fused_state
    if state.get("file"):
        fused_html.value = f"<span style='color:green;'>🔗 已融合: {state['file']}  scale={state['scale']}</span>"
    else:
        fused_html.value = "<span style='color:gray;'>未融合</span>"
    file_list_html.value = "<b>Lora 目录:</b><br>" + "".join(lines)

refresh_btn.on_click(lambda b: refresh_file_list())

# ── 执行逻辑 ──────────────────────────────────────────
def on_execute(b):
    output_widget.clear_output()
    execute_btn.disabled = True
    mode   = mode_dropdown.value
    method = download_method.value
    force  = force_check.value
    del_orig = del_orig_check.value
    scale  = scale_slider.value

    def log(msg):
        with output_widget:
            print(msg)

    try:
        status_html.value = "<span style='color:orange;'>⏳ 执行中...</span>"

        # ── 卸载 ──
        if mode == "unfuse":
            unfuse_lora(log=log)
            status_html.value = "<span style='color:green;'>✅ 卸载完成</span>"
            refresh_file_list()
            return

        # ── 下载 ──
        downloaded_path = None
        if "download" in mode:
            if method == "url":
                url = url_input.value.strip()
                if not url:
                    log("❌ 请输入 URL"); status_html.value = "❌"; return
                fname = filename_input.value.strip() or None
                log(f"⬇️ URL下载: {url}")
                local_path, result = download_from_url(url, fname, force)
            else:
                repo = repo_input.value.strip()
                fname = filename_input.value.strip()
                if not repo or not fname:
                    log("❌ 请填写 Repo ID 和文件名"); status_html.value = "❌"; return
                log(f"⬇️ Hub下载: {repo}/{fname}")
                local_path, result = download_from_hub(repo, fname, force)

            if local_path:
                log(f"{'✅ 下载成功' if result == 'downloaded' else '⏭️ 已存在'}: {local_path}")
                downloaded_path = local_path
            else:
                log(f"❌ 下载失败: {result}"); status_html.value = "❌"; return

        # ── 转换 ──
        converted_path = None
        if "convert" in mode:
            # 找到要转换的文件
            src = downloaded_path
            if src is None:
                # 纯转换模式：取目录里第一个需要转换的文件
                candidates = [
                    os.path.join(LORA_DIR, f)
                    for f in os.listdir(LORA_DIR) if f.endswith(".safetensors")
                ]
                needs = [f for f in candidates if needs_conversion(f)]
                if not needs:
                    log("✅ 没有需要转换的文件")
                else:
                    for src_f in needs:
                        log(f"\n🔧 转换: {Path(src_f).name}")
                        converted_path = convert_lora_file(src_f, del_orig, log)
            else:
                if needs_conversion(src):
                    log(f"\n🔧 转换: {Path(src).name}")
                    converted_path = convert_lora_file(src, del_orig, log)
                else:
                    log("⏭️ 该文件无需转换")
                    converted_path = src  # 已经是 diffusers 格式

        # ── 融合 ──
        if "fuse" in mode:
            # 找到要融合的文件
            fuse_src = converted_path or downloaded_path
            if fuse_src is None:
                # 从目录找最新的可用文件
                candidates = sorted([
                    os.path.join(LORA_DIR, f)
                    for f in os.listdir(LORA_DIR) if f.endswith(".safetensors")
                ], key=os.path.getmtime, reverse=True)
                available = [f for f in candidates if not needs_conversion(f)]
                if not available:
                    log("❌ 没有可融合的 diffusers 格式文件"); status_html.value = "❌"; return
                fuse_src = available[0]
            log(f"\n🔗 融合: {Path(fuse_src).name}  scale={scale:.2f}")
            fuse_lora(fuse_src, scale=scale, log=log)

        status_html.value = "<span style='color:green;'>✅ 完成</span>"
        refresh_file_list()

    except Exception as e:
        log(f"\n❌ 错误: {e}")
        import traceback; traceback.print_exc()
        status_html.value = "<span style='color:red;'>❌ 失败</span>"
    finally:
        execute_btn.disabled = False

execute_btn.on_click(on_execute)
refresh_file_list()

# ════════════════════════════════════════════════════════
# 6. 布局
# ════════════════════════════════════════════════════════
header = widgets.HTML("""
<div style='background:linear-gradient(135deg,#11998e,#38ef7d);
            color:white;padding:12px 16px;border-radius:8px;margin-bottom:8px;'>
  <h3 style='margin:0;'>🧩 LoRA 下载 / 转换 / 融合</h3>
  <span style='font-size:12px;'>URL / Repo 下载 | Kohya→Diffusers 转换 | 融合 & 卸载</span>
</div>
""")

ui = widgets.VBox([
    header,
    download_method,
    url_row,
    repo_row,
    filename_row,
    file_select_row,
    widgets.HBox([force_check, del_orig_check]),
    scale_slider,
    mode_dropdown,
    widgets.HBox([execute_btn, refresh_btn]),
    status_html,
    fused_html,
    file_list_html,
    output_widget,
], layout=widgets.Layout(padding="12px", border="1px solid #ddd",
                          border_radius="10px", width="100%", max_width="1100px"))

display(ui)

In [ ]:
# @title ⚡ DiT 缓存控制面板（不使用时一定要关闭）
import ipywidgets as widgets
from IPython.display import display

# ----------------------------
# UI 组件
# ----------------------------
cache_output = widgets.Output()

# ========== 统一方向说明 ==========
direction_banner = widgets.HTML(value="""
<div style='background:#f0f4ff; border-left:4px solid #2575fc; padding:8px 12px;
            margin-bottom:6px; border-radius:0 6px 6px 0; font-size:12px;'>
    🎯 <b>所有滑块方向统一</b>：← 左侧 = 质量优先（慢）&nbsp;&nbsp;|&nbsp;&nbsp;右侧 = 速度优先（快） →
</div>
""")

# ========== 残差阈值（原方向已正确：小=保守，大=激进） ==========
threshold_slider = widgets.FloatSlider(
    value=0.04,
    min=0.01,
    max=0.12,
    step=0.01,
    description="",
    readout_format=".2f",
    style={"description_width": "0px"},
    layout=widgets.Layout(width="420px"),
)
threshold_label = widgets.HTML(value="""
<div style='width:160px;'>
    <b style='font-size:13px;'>残差阈值</b><br>
    <span style='font-size:11px; color:#666;'>
        越小=越少跳过=质量高<br>越大=越多跳过=速度快
    </span>
</div>
""")
threshold_value_display = widgets.HTML(value="")

# ========== 缓存激进度（反向映射 warmup：激进度高 → warmup少） ==========
#   UI显示 1~10，含义"激进度"：1=最保守(warmup=10), 10=最激进(warmup=1)
#   实际 warmup = 11 - 滑块值
aggressive_warmup_slider = widgets.IntSlider(
    value=5,      # 对应 warmup=6? 不，我们用更直觉的映射
    min=1,
    max=10,
    step=1,
    description="",
    style={"description_width": "0px"},
    layout=widgets.Layout(width="420px"),
)
aggressive_warmup_label = widgets.HTML(value="""
<div style='width:160px;'>
    <b style='font-size:13px;'>前期跳过激进度</b><br>
    <span style='font-size:11px; color:#666;'>
        越小=前期保护越多=质量高<br>越大=前期保护越少=速度快
    </span>
</div>
""")
aggressive_warmup_hint = widgets.HTML(value="")

# ========== 尾部跳过激进度（反向映射 skip_last：激进度高 → skip少） ==========
#   UI显示 1~9，含义"激进度"：1=最保守(skip=8), 9=最激进(skip=0)
#   实际 skip_last = 9 - 滑块值
aggressive_tail_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=9,
    step=1,
    description="",
    style={"description_width": "0px"},
    layout=widgets.Layout(width="420px"),
)
aggressive_tail_label = widgets.HTML(value="""
<div style='width:160px;'>
    <b style='font-size:13px;'>尾部跳过激进度</b><br>
    <span style='font-size:11px; color:#666;'>
        越小=尾部保护越多=质量高<br>越大=尾部保护越少=速度快
    </span>
</div>
""")
aggressive_tail_hint = widgets.HTML(value="")

# ========== 反向映射函数 ==========
def get_actual_warmup():
    """UI激进度 → 实际warmup步数"""
    return 11 - aggressive_warmup_slider.value  # UI=1→warmup=10, UI=10→warmup=1

def get_actual_skip_last():
    """UI激进度 → 实际保护尾块数"""
    return 9 - aggressive_tail_slider.value     # UI=1→skip=8, UI=9→skip=0

# ========== 实时参数显示 ==========
param_display = widgets.HTML(value="")

def update_param_display(*args):
    warmup = get_actual_warmup()
    skip   = get_actual_skip_last()
    thresh = threshold_slider.value

    # 综合评估
    score = thresh * 100 + aggressive_warmup_slider.value * 2 + aggressive_tail_slider.value * 2
    if score < 25:
        level, color, emoji = "质量优先", "#2e7d32", "🎨"
    elif score < 45:
        level, color, emoji = "均衡模式", "#f57c00", "⚖️"
    else:
        level, color, emoji = "速度优先", "#c62828", "🚀"

    param_display.value = f"""
    <div style='background:#fafafa; border:1px solid #e0e0e0; border-radius:6px;
                padding:8px 14px; margin-top:4px; font-size:12px;'>
        <b>📐 实际参数</b>：阈值={thresh:.2f}　|　warmup={warmup}步　|　保护尾块={skip}块
        <span style='float:right; color:{color}; font-weight:bold;'>{emoji} {level}</span>
    </div>
    """

threshold_slider.observe(update_param_display, names="value")
aggressive_warmup_slider.observe(update_param_display, names="value")
aggressive_tail_slider.observe(update_param_display, names="value")
update_param_display()  # 初始化

# ========== 启用/禁用开关 ==========
enable_toggle = widgets.ToggleButton(
    value=False,
    description="缓存已禁用",
    button_style="danger",
    icon="times",
    layout=widgets.Layout(width="140px", height="36px"),
)

# ========== 操作按钮 ==========
apply_btn = widgets.Button(
    description="🚀 应用配置",
    button_style="primary",
    layout=widgets.Layout(width="120px", height="36px"),
)
stats_btn = widgets.Button(
    description="📊 查看统计",
    button_style="info",
    layout=widgets.Layout(width="120px", height="36px"),
)
reset_btn = widgets.Button(
    description="🔄 重置计数",
    button_style="warning",
    layout=widgets.Layout(width="120px", height="36px"),
)

cache_status_html = widgets.HTML(value="<span style='color:gray;'>就绪</span>")

# ========== 预设快捷按钮 ==========
preset_html = widgets.HTML(value="<b style='font-size:12px;'>快捷预设：</b>")
preset_quality_btn = widgets.Button(
    description="🎨 质量优先",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_balance_btn = widgets.Button(
    description="⚖️ 均衡",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_speed_btn = widgets.Button(
    description="🚀 速度优先",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_off_btn = widgets.Button(
    description="⛔ 关闭缓存",
    button_style="danger",
    layout=widgets.Layout(width="110px", height="30px"),
)

# ----------------------------
# 预设配置（统一用 UI 激进度语义）
# ----------------------------
PRESETS = {
    #                  threshold  UI激进度_warmup  UI激进度_tail
    "quality": {"threshold": 0.02, "agg_warmup": 2,  "agg_tail": 2,  "label": "质量优先", "expect": "~10%跳过"},
    "balance": {"threshold": 0.04, "agg_warmup": 5,  "agg_tail": 5,  "label": "均衡",     "expect": "~30%跳过"},
    "speed":   {"threshold": 0.08, "agg_warmup": 8,  "agg_tail": 8,  "label": "速度优先", "expect": "~40%跳过"},
}

def apply_preset(name):
    p = PRESETS[name]
    threshold_slider.value         = p["threshold"]
    aggressive_warmup_slider.value = p["agg_warmup"]
    aggressive_tail_slider.value   = p["agg_tail"]
    cache_status_html.value = f"<span style='color:#9932CC;'>📌 预设：{p['label']}（预期 {p['expect']}）</span>"

preset_quality_btn.on_click(lambda b: apply_preset("quality"))
preset_balance_btn.on_click(lambda b: apply_preset("balance"))
preset_speed_btn.on_click(lambda b: apply_preset("speed"))

def on_preset_off(b):
    enable_toggle.value = False
    enable_toggle.description = "缓存已禁用"
    enable_toggle.button_style = "danger"
    enable_toggle.icon = "times"
    on_apply_click(None)

preset_off_btn.on_click(on_preset_off)

# ----------------------------
# Toggle 回调
# ----------------------------
def on_toggle_change(change):
    if change["new"]:
        enable_toggle.description = "缓存已启用"
        enable_toggle.button_style = "success"
        enable_toggle.icon = "check"
    else:
        enable_toggle.description = "缓存已禁用"
        enable_toggle.button_style = "danger"
        enable_toggle.icon = "times"

enable_toggle.observe(on_toggle_change, names="value")

# ----------------------------
# 应用配置（核心：从 UI 激进度反算实际参数）
# ----------------------------
def on_apply_click(b):
    cache_output.clear_output()
    apply_btn.disabled = True

    with cache_output:
        try:
            frame_globals = {}
            try:
                import IPython
                shell = IPython.get_ipython()
                frame_globals = shell.user_ns
            except Exception:
                pass

            dit_cache = frame_globals.get("_dit_cache_instance")
            _pipe     = frame_globals.get("pipe")

            if not enable_toggle.value:
                if dit_cache is not None:
                    dit_cache.disable()
                    frame_globals["_dit_cache_instance"] = None
                    print("⛔ 缓存已禁用，forward 已还原")
                    cache_status_html.value = "<span style='color:red;'>⛔ 缓存已禁用</span>"
                else:
                    print("ℹ️ 缓存本就未启用")
                    cache_status_html.value = "<span style='color:gray;'>ℹ️ 未启用</span>"
                return

            if _pipe is None:
                print("❌ pipe 未加载，请先在模型管理器中加载模型")
                cache_status_html.value = "<span style='color:red;'>❌ pipe 未就绪</span>"
                return

            # ★ 反向映射：从 UI 激进度 → 实际参数
            threshold = threshold_slider.value
            warmup    = get_actual_warmup()
            skip_last = get_actual_skip_last()

            if dit_cache is not None:
                dit_cache.disable()
                print("🔄 旧缓存已禁用")

            new_cache = ZImageDiTCache(
                pipe=_pipe,
                residual_diff_threshold=threshold,
                warmup_steps=warmup,
                skip_last_n_blocks=skip_last,
            )
            new_cache.enable()
            frame_globals["_dit_cache_instance"] = new_cache

            print(f"\n⚙️  配置摘要：")
            print(f"   残差阈值：  {threshold}")
            print(f"   Warmup步数：{warmup}（激进度 {aggressive_warmup_slider.value}/10）")
            print(f"   保护尾块：  {skip_last}（激进度 {aggressive_tail_slider.value}/9）")
            cache_status_html.value = (
                f"<span style='color:green;'>✅ 已应用｜阈值={threshold} warmup={warmup} 保护={skip_last}块</span>"
            )

        except NameError as e:
            print(f"❌ 找不到 ZImageDiTCache 类：{e}")
            print("请确认模型管理器单元格已运行")
            cache_status_html.value = "<span style='color:red;'>❌ 请先运行模型管理器</span>"
        except Exception as e:
            print(f"❌ 应用失败：{e}")
            import traceback; traceback.print_exc()
            cache_status_html.value = "<span style='color:red;'>❌ 失败</span>"
        finally:
            apply_btn.disabled = False

apply_btn.on_click(on_apply_click)

# ----------------------------
# 统计
# ----------------------------
def on_stats_click(b):
    cache_output.clear_output()
    with cache_output:
        try:
            import IPython
            shell = IPython.get_ipython()
            dit_cache = shell.user_ns.get("_dit_cache_instance")

            if dit_cache is None:
                print("ℹ️ 缓存未启用")
                return
            dit_cache.stats()

            print("\n📋 各块命中明细（前10块）：")
            for i, block in enumerate(dit_cache.pipe.transformer.layers[:10]):
                cs = getattr(block, "_cache_state", None)
                if cs:
                    total = cs["hit_count"] + cs["miss_count"]
                    rate  = cs["hit_count"] / total * 100 if total > 0 else 0
                    print(f"   layer[{i:2d}]  命中={cs['hit_count']:3d}  计算={cs['miss_count']:3d}  命中率={rate:.0f}%")

        except Exception as e:
            print(f"❌ 统计失败：{e}")

stats_btn.on_click(on_stats_click)

# ----------------------------
# 重置计数
# ----------------------------
def on_reset_click(b):
    cache_output.clear_output()
    with cache_output:
        try:
            import IPython
            shell = IPython.get_ipython()
            dit_cache = shell.user_ns.get("_dit_cache_instance")

            if dit_cache is None:
                print("ℹ️ 缓存未启用，无需重置")
                return
            dit_cache.reset()
            print("🔄 计数器已重置，下次推理将重新统计")
            cache_status_html.value = "<span style='color:orange;'>🔄 已重置，等待推理...</span>"
        except Exception as e:
            print(f"❌ 重置失败：{e}")

reset_btn.on_click(on_reset_click)

# ----------------------------
# UI 布局
# ----------------------------
cache_header = widgets.HTML(value="""
<div style='background:linear-gradient(135deg,#6a11cb 0%,#2575fc 100%);
            color:white; padding:12px 15px; border-radius:8px; margin-bottom:10px;'>
    <h3 style='margin:0;'>⚡ DiT 缓存控制面板</h3>
    <span style='font-size:12px;'>ZImageDiTCache | 残差跳过加速 | 仅适用于纯 INT8 模型</span>
</div>
""")

# 带标签的滑块行
slider_row_1 = widgets.HBox([threshold_label,         threshold_slider],
                            layout=widgets.Layout(align_items="center", gap="10px"))
slider_row_2 = widgets.HBox([aggressive_warmup_label, aggressive_warmup_slider],
                            layout=widgets.Layout(align_items="center", gap="10px"))
slider_row_3 = widgets.HBox([aggressive_tail_label,   aggressive_tail_slider],
                            layout=widgets.Layout(align_items="center", gap="10px"))

# 左右标注
slider_direction = widgets.HTML(value="""
<div style='margin-left:170px; width:420px; display:flex; justify-content:space-between;
            font-size:11px; color:#888; margin-top:-2px; margin-bottom:6px;'>
    <span>🎨 质量优先</span>
    <span>🚀 速度优先</span>
</div>
""")

cache_layout = widgets.VBox([
    cache_header,

    # 预设行
    widgets.HBox([
        preset_html,
        preset_quality_btn,
        preset_balance_btn,
        preset_speed_btn,
        preset_off_btn,
    ], layout=widgets.Layout(align_items="center", gap="6px")),

    widgets.HTML(value="<hr style='margin:8px 0;'>"),

    direction_banner,

    # 参数滑块（统一方向）
    widgets.VBox([
        slider_row_1,
        slider_row_2,
        slider_row_3,
        slider_direction,
        param_display,
    ], layout=widgets.Layout(gap="6px")),

    widgets.HTML(value="<hr style='margin:8px 0;'>"),

    # 操作行
    widgets.HBox([
        enable_toggle,
        apply_btn,
        stats_btn,
        reset_btn,
        cache_status_html,
    ], layout=widgets.Layout(align_items="center", gap="8px")),

    cache_output,

], layout=widgets.Layout(
    padding="10px",
    border="1px solid #ddd",
    border_radius="10px",
    width="760px",
))

display(cache_layout)

In [ ]:
# @title ⚡ DiT 缓存控制面板（不使用时一定要关闭）
import ipywidgets as widgets
from IPython.display import display

# ----------------------------
# UI 组件
# ----------------------------
cache_output = widgets.Output()

# ========== 统一方向说明 ==========
direction_banner = widgets.HTML(value="""
<div style='background:#f0f4ff; border-left:4px solid #2575fc; padding:8px 12px;
            margin-bottom:6px; border-radius:0 6px 6px 0; font-size:12px;'>
    🎯 <b>所有滑块方向统一</b>：← 左侧 = 质量优先（慢）&nbsp;&nbsp;|&nbsp;&nbsp;右侧 = 速度优先（快） →
</div>
""")

# ========== 残差阈值 ==========
threshold_slider = widgets.FloatSlider(
    value=0.04,
    min=0.01,
    max=0.12,
    step=0.01,
    description="",
    readout_format=".2f",
    style={"description_width": "0px"},
    layout=widgets.Layout(width="420px"),
)
threshold_label = widgets.HTML(value="""
<div style='width:160px;'>
    <b style='font-size:13px;'>残差阈值</b><br>
    <span style='font-size:11px; color:#666;'>
        越小=越少跳过=质量高<br>越大=越多跳过=速度快
    </span>
</div>
""")

# ========== 缓存激进度（反向映射 warmup） ==========
aggressive_warmup_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    step=1,
    description="",
    style={"description_width": "0px"},
    layout=widgets.Layout(width="420px"),
)
aggressive_warmup_label = widgets.HTML(value="""
<div style='width:160px;'>
    <b style='font-size:13px;'>前期跳过激进度</b><br>
    <span style='font-size:11px; color:#666;'>
        越小=前期保护越多=质量高<br>越大=前期保护越少=速度快
    </span>
</div>
""")

# ========== 尾部跳过激进度（反向映射 skip_last） ==========
aggressive_tail_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=9,
    step=1,
    description="",
    style={"description_width": "0px"},
    layout=widgets.Layout(width="420px"),
)
aggressive_tail_label = widgets.HTML(value="""
<div style='width:160px;'>
    <b style='font-size:13px;'>尾部跳过激进度</b><br>
    <span style='font-size:11px; color:#666;'>
        越小=尾部保护越多=质量高<br>越大=尾部保护越少=速度快
    </span>
</div>
""")

# ========== 反向映射函数 ==========
def get_actual_warmup():
    """UI激进度 → 实际warmup步数"""
    return 11 - aggressive_warmup_slider.value

def get_actual_skip_last():
    """UI激进度 → 实际保护尾块数"""
    return 9 - aggressive_tail_slider.value

# ========== 实时参数显示（简化版：只显示实际参数） ==========
param_display = widgets.HTML(value="")

def update_param_display(*args):
    warmup = get_actual_warmup()
    skip = get_actual_skip_last()
    thresh = threshold_slider.value

    param_display.value = f"""
    <div style='background:#fafafa; border:1px solid #e0e0e0; border-radius:6px;
                padding:8px 14px; margin-top:4px; font-size:12px;'>
        <b>📐 实际参数</b>：阈值={thresh:.2f}　|　warmup={warmup}步　|　保护尾块={skip}块
    </div>
    """

threshold_slider.observe(update_param_display, names="value")
aggressive_warmup_slider.observe(update_param_display, names="value")
aggressive_tail_slider.observe(update_param_display, names="value")
update_param_display()

# ========== 启用/禁用开关 ==========
enable_toggle = widgets.ToggleButton(
    value=False,
    description="缓存已禁用",
    button_style="danger",
    icon="times",
    layout=widgets.Layout(width="140px", height="36px"),
)

# ========== 操作按钮 ==========
apply_btn = widgets.Button(
    description="🚀 应用配置",
    button_style="primary",
    layout=widgets.Layout(width="120px", height="36px"),
)
stats_btn = widgets.Button(
    description="📊 查看统计",
    button_style="info",
    layout=widgets.Layout(width="120px", height="36px"),
)
reset_btn = widgets.Button(
    description="🔄 重置计数",
    button_style="warning",
    layout=widgets.Layout(width="120px", height="36px"),
)

cache_status_html = widgets.HTML(value="<span style='color:gray;'>就绪</span>")

# ========== 预设快捷按钮 ==========
preset_html = widgets.HTML(value="<b style='font-size:12px;'>快捷预设：</b>")
preset_quality_btn = widgets.Button(
    description="🎨 质量优先",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_balance_btn = widgets.Button(
    description="⚖️ 均衡",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_speed_btn = widgets.Button(
    description="🚀 速度优先",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_off_btn = widgets.Button(
    description="⛔ 关闭缓存",
    button_style="danger",
    layout=widgets.Layout(width="110px", height="30px"),
)

# ----------------------------
# 预设配置
# ----------------------------
PRESETS = {
    "quality": {"threshold": 0.02, "agg_warmup": 2, "agg_tail": 2, "label": "🎨 质量优先", "expect": "~10%跳过"},
    "balance": {"threshold": 0.04, "agg_warmup": 5, "agg_tail": 5, "label": "⚖️ 均衡",    "expect": "~30%跳过"},
    "speed":   {"threshold": 0.08, "agg_warmup": 8, "agg_tail": 8, "label": "🚀 速度优先", "expect": "~40%跳过"},
}

def apply_preset(name):
    p = PRESETS[name]
    threshold_slider.value = p["threshold"]
    aggressive_warmup_slider.value = p["agg_warmup"]
    aggressive_tail_slider.value = p["agg_tail"]
    cache_status_html.value = f"<span style='color:#9932CC;'>📌 已选预设：{p['label']}（预期 {p['expect']}）</span>"

preset_quality_btn.on_click(lambda b: apply_preset("quality"))
preset_balance_btn.on_click(lambda b: apply_preset("balance"))
preset_speed_btn.on_click(lambda b: apply_preset("speed"))

def on_preset_off(b):
    enable_toggle.value = False
    enable_toggle.description = "缓存已禁用"
    enable_toggle.button_style = "danger"
    enable_toggle.icon = "times"
    on_apply_click(None)

preset_off_btn.on_click(on_preset_off)

# ----------------------------
# Toggle 回调
# ----------------------------
def on_toggle_change(change):
    if change["new"]:
        enable_toggle.description = "缓存已启用"
        enable_toggle.button_style = "success"
        enable_toggle.icon = "check"
    else:
        enable_toggle.description = "缓存已禁用"
        enable_toggle.button_style = "danger"
        enable_toggle.icon = "times"

enable_toggle.observe(on_toggle_change, names="value")

# ----------------------------
# 应用配置
# ----------------------------
def on_apply_click(b):
    cache_output.clear_output()
    apply_btn.disabled = True

    with cache_output:
        try:
            frame_globals = {}
            try:
                import IPython
                shell = IPython.get_ipython()
                frame_globals = shell.user_ns
            except Exception:
                pass

            dit_cache = frame_globals.get("_dit_cache_instance")
            _pipe = frame_globals.get("pipe")

            if not enable_toggle.value:
                if dit_cache is not None:
                    dit_cache.disable()
                    frame_globals["_dit_cache_instance"] = None
                    print("⛔ 缓存已禁用，forward 已还原")
                    cache_status_html.value = "<span style='color:red;'>⛔ 缓存已禁用</span>"
                else:
                    print("ℹ️ 缓存本就未启用")
                    cache_status_html.value = "<span style='color:gray;'>ℹ️ 未启用</span>"
                return

            if _pipe is None:
                print("❌ pipe 未加载，请先在模型管理器中加载模型")
                cache_status_html.value = "<span style='color:red;'>❌ pipe 未就绪</span>"
                return

            # 反向映射：从 UI 激进度 → 实际参数
            threshold = threshold_slider.value
            warmup = get_actual_warmup()
            skip_last = get_actual_skip_last()

            if dit_cache is not None:
                dit_cache.disable()
                print("🔄 旧缓存已禁用")

            new_cache = ZImageDiTCache(
                pipe=_pipe,
                residual_diff_threshold=threshold,
                warmup_steps=warmup,
                skip_last_n_blocks=skip_last,
            )
            new_cache.enable()
            frame_globals["_dit_cache_instance"] = new_cache

            print(f"\n⚙️ 配置已应用：")
            print(f"   残差阈值：{threshold}")
            print(f"   Warmup：{warmup} 步")
            print(f"   保护尾块：{skip_last} 块")

            cache_status_html.value = f"<span style='color:green;'>✅ 已应用</span>"

        except NameError as e:
            print(f"❌ 找不到 ZImageDiTCache 类：{e}")
            print("请确认模型管理器单元格已运行")
            cache_status_html.value = "<span style='color:red;'>❌ 请先运行模型管理器</span>"
        except Exception as e:
            print(f"❌ 应用失败：{e}")
            import traceback
            traceback.print_exc()
            cache_status_html.value = "<span style='color:red;'>❌ 失败</span>"
        finally:
            apply_btn.disabled = False

apply_btn.on_click(on_apply_click)

# ----------------------------
# 统计
# ----------------------------
def on_stats_click(b):
    cache_output.clear_output()
    with cache_output:
        try:
            import IPython
            shell = IPython.get_ipython()
            dit_cache = shell.user_ns.get("_dit_cache_instance")

            if dit_cache is None:
                print("ℹ️ 缓存未启用")
                return
            dit_cache.stats()

            print("\n📋 各块命中明细（前10块）：")
            for i, block in enumerate(dit_cache.pipe.transformer.layers[:10]):
                cs = getattr(block, "_cache_state", None)
                if cs:
                    total = cs["hit_count"] + cs["miss_count"]
                    rate = cs["hit_count"] / total * 100 if total > 0 else 0
                    print(f"   layer[{i:2d}]  命中={cs['hit_count']:3d}  计算={cs['miss_count']:3d}  命中率={rate:.0f}%")

        except Exception as e:
            print(f"❌ 统计失败：{e}")

stats_btn.on_click(on_stats_click)

# ----------------------------
# 重置计数
# ----------------------------
def on_reset_click(b):
    cache_output.clear_output()
    with cache_output:
        try:
            import IPython
            shell = IPython.get_ipython()
            dit_cache = shell.user_ns.get("_dit_cache_instance")

            if dit_cache is None:
                print("ℹ️ 缓存未启用，无需重置")
                return
            dit_cache.reset()
            print("🔄 计数器已重置，下次推理将重新统计")
            cache_status_html.value = "<span style='color:orange;'>🔄 已重置</span>"
        except Exception as e:
            print(f"❌ 重置失败：{e}")

reset_btn.on_click(on_reset_click)

# ----------------------------
# UI 布局
# ----------------------------
cache_header = widgets.HTML(value="""
<div style='background:linear-gradient(135deg,#6a11cb 0%,#2575fc 100%);
            color:white; padding:12px 15px; border-radius:8px; margin-bottom:10px;'>
    <h3 style='margin:0;'>⚡ DiT 缓存控制面板</h3>
    <span style='font-size:12px;'>ZImageDiTCache | 残差跳过加速 | 仅适用于纯 INT8 模型</span>
</div>
""")

# 带标签的滑块行
slider_row_1 = widgets.HBox([threshold_label, threshold_slider],
                            layout=widgets.Layout(align_items="center", gap="10px"))
slider_row_2 = widgets.HBox([aggressive_warmup_label, aggressive_warmup_slider],
                            layout=widgets.Layout(align_items="center", gap="10px"))
slider_row_3 = widgets.HBox([aggressive_tail_label, aggressive_tail_slider],
                            layout=widgets.Layout(align_items="center", gap="10px"))

# 左右标注
slider_direction = widgets.HTML(value="""
<div style='margin-left:170px; width:420px; display:flex; justify-content:space-between;
            font-size:11px; color:#888; margin-top:-2px; margin-bottom:6px;'>
    <span>🎨 质量优先</span>
    <span>🚀 速度优先</span>
</div>
""")

cache_layout = widgets.VBox([
    cache_header,

    # 预设行
    widgets.HBox([
        preset_html,
        preset_quality_btn,
        preset_balance_btn,
        preset_speed_btn,
        preset_off_btn,
    ], layout=widgets.Layout(align_items="center", gap="6px")),

    widgets.HTML(value="<hr style='margin:8px 0;'>"),

    direction_banner,

    # 参数滑块（统一方向）
    widgets.VBox([
        slider_row_1,
        slider_row_2,
        slider_row_3,
        slider_direction,
        param_display,
    ], layout=widgets.Layout(gap="6px")),

    widgets.HTML(value="<hr style='margin:8px 0;'>"),

    # 操作行
    widgets.HBox([
        enable_toggle,
        apply_btn,
        stats_btn,
        reset_btn,
        cache_status_html,
    ], layout=widgets.Layout(align_items="center", gap="8px")),

    cache_output,

], layout=widgets.Layout(
    padding="10px",
    border="1px solid #ddd",
    border_radius="10px",
    width="760px",
))

display(cache_layout)

In [ ]:
# @title ⚡ DiT 缓存控制面板（旧版）
import ipywidgets as widgets
from IPython.display import display

# ----------------------------
# UI 组件
# ----------------------------
cache_output = widgets.Output()

# 阈值滑块
threshold_slider = widgets.FloatSlider(
    value=0.04,
    min=0.01,
    max=0.12,
    step=0.01,
    description="残差阈值:",
    readout_format=".2f",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="480px"),
)
threshold_hint = widgets.HTML(
    value="<span style='font-size:11px;color:#888;'>← 保守（质量优先）&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;激进（速度优先）→</span>"
)

# warmup 步数
warmup_slider = widgets.IntSlider(
    value=2,
    min=1,
    max=10,
    step=1,
    description="Warmup步:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="480px"),
)
warmup_hint = widgets.HTML(
    value="<span style='font-size:11px;color:#888;'>前N步完全不缓存（去噪早期结构变化剧烈）</span>"
)

# 跳过尾部块数
skip_slider = widgets.IntSlider(
    value=2,
    min=0,
    max=8,
    step=1,
    description="保护尾块:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="480px"),
)
skip_hint = widgets.HTML(
    value="<span style='font-size:11px;color:#888;'>最后N块不缓存（靠近输出层，语义敏感）</span>"
)

# 启用/禁用开关
enable_toggle = widgets.ToggleButton(
    value=True,
    description="缓存已启用",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="140px", height="36px"),
)

# 应用按钮
apply_btn = widgets.Button(
    description="🚀 应用配置",
    button_style="primary",
    layout=widgets.Layout(width="120px", height="36px"),
)

# 统计按钮
stats_btn = widgets.Button(
    description="📊 查看统计",
    button_style="info",
    layout=widgets.Layout(width="120px", height="36px"),
)

# 重置按钮
reset_btn = widgets.Button(
    description="🔄 重置计数",
    button_style="warning",
    layout=widgets.Layout(width="120px", height="36px"),
)

cache_status_html = widgets.HTML(value="<span style='color:gray;'>就绪</span>")

# 预设快捷按钮
preset_html = widgets.HTML(value="<b style='font-size:12px;'>快捷预设：</b>")
preset_quality_btn = widgets.Button(
    description="🎨 质量优先",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_balance_btn = widgets.Button(
    description="⚖️ 均衡",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_speed_btn = widgets.Button(
    description="🚀 速度优先",
    button_style="",
    layout=widgets.Layout(width="110px", height="30px"),
)
preset_off_btn = widgets.Button(
    description="⛔ 关闭缓存",
    button_style="danger",
    layout=widgets.Layout(width="110px", height="30px"),
)

# ----------------------------
# 预设配置
# ----------------------------
PRESETS = {
    "quality": {"threshold": 0.02, "warmup": 4, "skip": 4,  "label": "质量优先",  "expect": "~10%跳过"},
    "balance": {"threshold": 0.04, "warmup": 2, "skip": 2,  "label": "均衡",      "expect": "~30%跳过"},
    "speed":   {"threshold": 0.08, "warmup": 1, "skip": 1,  "label": "速度优先",  "expect": "~40%跳过"},
}

def apply_preset(name):
    p = PRESETS[name]
    threshold_slider.value = p["threshold"]
    warmup_slider.value    = p["warmup"]
    skip_slider.value      = p["skip"]
    cache_status_html.value = f"<span style='color:#9932CC;'>📌 预设：{p['label']}（预期 {p['expect']}）</span>"

preset_quality_btn.on_click(lambda b: apply_preset("quality"))
preset_balance_btn.on_click(lambda b: apply_preset("balance"))
preset_speed_btn.on_click(lambda b: apply_preset("speed"))

def on_preset_off(b):
    enable_toggle.value = False
    enable_toggle.description = "缓存已禁用"
    enable_toggle.button_style = "danger"
    enable_toggle.icon = "times"
    on_apply_click(None)

preset_off_btn.on_click(on_preset_off)

# ----------------------------
# Toggle 回调
# ----------------------------
def on_toggle_change(change):
    if change["new"]:
        enable_toggle.description = "缓存已启用"
        enable_toggle.button_style = "success"
        enable_toggle.icon = "check"
    else:
        enable_toggle.description = "缓存已禁用"
        enable_toggle.button_style = "danger"
        enable_toggle.icon = "times"

enable_toggle.observe(on_toggle_change, names="value")

# ----------------------------
# 应用配置
# ----------------------------
def on_apply_click(b):
    cache_output.clear_output()
    apply_btn.disabled = True

    with cache_output:
        try:
            # 检查全局变量
            g = globals()
            import builtins
            frame_globals = {}
            try:
                import IPython
                shell = IPython.get_ipython()
                frame_globals = shell.user_ns
            except Exception:
                pass

            dit_cache = frame_globals.get("_dit_cache_instance")
            _pipe     = frame_globals.get("pipe")

            if not enable_toggle.value:
                # 禁用缓存
                if dit_cache is not None:
                    dit_cache.disable()
                    frame_globals["_dit_cache_instance"] = None
                    print("⛔ 缓存已禁用，forward 已还原")
                    cache_status_html.value = "<span style='color:red;'>⛔ 缓存已禁用</span>"
                else:
                    print("ℹ️ 缓存本就未启用")
                    cache_status_html.value = "<span style='color:gray;'>ℹ️ 未启用</span>"
                return

            if _pipe is None:
                print("❌ pipe 未加载，请先在模型管理器中加载模型")
                cache_status_html.value = "<span style='color:red;'>❌ pipe 未就绪</span>"
                return

            threshold = threshold_slider.value
            warmup    = warmup_slider.value
            skip_last = skip_slider.value

            # 禁用旧缓存
            if dit_cache is not None:
                dit_cache.disable()
                print("🔄 旧缓存已禁用")

            # 重建实例
            new_cache = ZImageDiTCache(
                pipe=_pipe,
                residual_diff_threshold=threshold,
                warmup_steps=warmup,
                skip_last_n_blocks=skip_last,
            )
            new_cache.enable()
            frame_globals["_dit_cache_instance"] = new_cache

            print(f"\n⚙️  配置摘要：")
            print(f"   残差阈值：{threshold}")
            print(f"   Warmup：  {warmup} 步")
            print(f"   保护尾块：{skip_last} 块")
            cache_status_html.value = (
                f"<span style='color:green;'>✅ 已应用｜阈值={threshold} warmup={warmup} 保护={skip_last}块</span>"
            )

        except NameError as e:
            print(f"❌ 找不到 ZImageDiTCache 类：{e}")
            print("请确认模型管理器单元格已运行")
            cache_status_html.value = "<span style='color:red;'>❌ 请先运行模型管理器</span>"
        except Exception as e:
            print(f"❌ 应用失败：{e}")
            import traceback; traceback.print_exc()
            cache_status_html.value = "<span style='color:red;'>❌ 失败</span>"
        finally:
            apply_btn.disabled = False

apply_btn.on_click(on_apply_click)

# ----------------------------
# 统计
# ----------------------------
def on_stats_click(b):
    cache_output.clear_output()
    with cache_output:
        try:
            import IPython
            shell = IPython.get_ipython()
            dit_cache = shell.user_ns.get("_dit_cache_instance")

            if dit_cache is None:
                print("ℹ️ 缓存未启用")
                return
            dit_cache.stats()

            # 额外显示每块命中详情
            print("\n📋 各块命中明细（前10块）：")
            for i, block in enumerate(dit_cache.pipe.transformer.layers[:10]):
                cs = getattr(block, "_cache_state", None)
                if cs:
                    total = cs["hit_count"] + cs["miss_count"]
                    rate  = cs["hit_count"] / total * 100 if total > 0 else 0
                    print(f"   layer[{i:2d}]  命中={cs['hit_count']:3d}  计算={cs['miss_count']:3d}  命中率={rate:.0f}%")

        except Exception as e:
            print(f"❌ 统计失败：{e}")

stats_btn.on_click(on_stats_click)

# ----------------------------
# 重置计数
# ----------------------------
def on_reset_click(b):
    cache_output.clear_output()
    with cache_output:
        try:
            import IPython
            shell = IPython.get_ipython()
            dit_cache = shell.user_ns.get("_dit_cache_instance")

            if dit_cache is None:
                print("ℹ️ 缓存未启用，无需重置")
                return
            dit_cache.reset()
            print("🔄 计数器已重置，下次推理将重新统计")
            cache_status_html.value = "<span style='color:orange;'>🔄 已重置，等待推理...</span>"
        except Exception as e:
            print(f"❌ 重置失败：{e}")

reset_btn.on_click(on_reset_click)

# ----------------------------
# UI 布局
# ----------------------------
cache_header = widgets.HTML(value="""
<div style='background:linear-gradient(135deg,#6a11cb 0%,#2575fc 100%);
            color:white; padding:12px 15px; border-radius:8px; margin-bottom:10px;'>
    <h3 style='margin:0;'>⚡ DiT 缓存控制面板</h3>
    <span style='font-size:12px;'>ZImageDiTCache | 残差跳过加速 | 仅适用于纯 INT8 模型</span>
</div>
""")

cache_layout = widgets.VBox([
    cache_header,

    # 预设行
    widgets.HBox([
        preset_html,
        preset_quality_btn,
        preset_balance_btn,
        preset_speed_btn,
        preset_off_btn,
    ], layout=widgets.Layout(align_items="center", gap="6px")),

    widgets.HTML(value="<hr style='margin:8px 0;'>"),

    # 参数滑块
    widgets.VBox([
        threshold_slider, threshold_hint,
        warmup_slider,    warmup_hint,
        skip_slider,      skip_hint,
    ], layout=widgets.Layout(gap="2px")),

    widgets.HTML(value="<hr style='margin:8px 0;'>"),

    # 操作行
    widgets.HBox([
        enable_toggle,
        apply_btn,
        stats_btn,
        reset_btn,
        cache_status_html,
    ], layout=widgets.Layout(align_items="center", gap="8px")),

    cache_output,

], layout=widgets.Layout(
    padding="10px",
    border="1px solid #ddd",
    border_radius="10px",
    width="760px",
))

display(cache_layout)

In [ ]:
# @title 🎛️ Z-Image 出图面板（采样器目前暂时仅Euler基线可用）
import os, gc, time, json, numpy as np, torch, traceback, inspect
from datetime import datetime
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab import drive

# ========= 依赖：模型管理器的 3 个函数 =========
def _require_manager_funcs():
    needed = ["ensure_tokenizer_loaded", "ensure_text_encoder_loaded", "drop_text_encoder"]
    missing = [n for n in needed if n not in globals()]
    if missing:
        raise RuntimeError("缺少模型管理器函数（先运行模型管理器并加载模型）: " + ", ".join(missing))
    return (globals()["ensure_tokenizer_loaded"],
            globals()["ensure_text_encoder_loaded"],
            globals()["drop_text_encoder"])

def _get_existing_pipe():
    if "pipe" in globals() and globals().get("pipe", None) is not None:
        return globals()["pipe"]
    if "loaded_components" in globals() and isinstance(globals()["loaded_components"], dict):
        return globals()["loaded_components"].get("pipe", None)
    return None

def _round_to_16(x: int) -> int:
    return int(round(x / 16) * 16)

def _pipe_accepts_kw(p, key: str) -> bool:
    try:
        sig = inspect.signature(p.__call__)
        return key in sig.parameters
    except Exception:
        return False

# ========= allocator/显存小工具 =========
def _get_allocator_conf():
    return os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "")

def _allocator_status_text():
    conf = _get_allocator_conf()
    if not conf:
        return ("<span style='color:orange;'>⚠️ allocator: 未设置 PYTORCH_CUDA_ALLOC_CONF</span>"
                "<br><span style='color:#64748b;font-size:12px;'>建议重启 runtime 后最先设置："
                "expandable_segments:True,max_split_size_mb:128</span>")
    ok = ("expandable_segments:True" in conf)
    if ok:
        return (f"<span style='color:green;'>✅ allocator: {conf}</span>"
                "<br><span style='color:#64748b;font-size:12px;'>已启用 expandable_segments（更抗碎片，适合冲大图）</span>")
    return (f"<span style='color:orange;'>⚠️ allocator: {conf}</span>"
            "<br><span style='color:#64748b;font-size:12px;'>建议包含 expandable_segments:True（需要重启后生效）</span>")

def _print_cuda_mem_line(tag=""):
    if not torch.cuda.is_available():
        print("CUDA 不可用")
        return
    alloc = torch.cuda.memory_allocated() / (1024**3)
    resv  = torch.cuda.memory_reserved() / (1024**3)
    peak  = torch.cuda.max_memory_allocated() / (1024**3)
    if tag:
        print(f"[{tag}] ", end="")
    print(f"allocated={alloc:.2f} GB | reserved={resv:.2f} GB | peak={peak:.2f} GB")

def runtime_fragmentation_cleanup(synchronize=True, reset_peak=False):
    if not torch.cuda.is_available():
        return
    if synchronize:
        try: torch.cuda.synchronize()
        except Exception: pass
    try: gc.collect()
    except Exception: pass
    try: torch.cuda.empty_cache()
    except Exception: pass
    try: torch.cuda.ipc_collect()
    except Exception: pass
    if reset_peak:
        try: torch.cuda.reset_peak_memory_stats()
        except Exception: pass
    if synchronize:
        try: torch.cuda.synchronize()
        except Exception: pass

# ========= SDPA 上下文 =========
def _enter_sdpa_efficient():
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
        ctx = sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH])
        ctx.__enter__()
        return ctx
    except Exception:
        return None

def _exit_sdpa(ctx):
    if ctx is None:
        return
    try:
        ctx.__exit__(None, None, None)
    except Exception:
        pass

# ========= dtype / 小工具 =========
def _dtype_from_mode(mode: str):
    mode = (mode or "fp32").lower()
    if mode == "fp16":
        return torch.float16
    if mode == "bf16":
        return torch.bfloat16
    return torch.float32

def _bf16_is_supported_cuda():
    if not torch.cuda.is_available():
        return False
    try:
        return bool(torch.cuda.is_bf16_supported())
    except Exception:
        return False

def _maybe_cast_pipe_modules(p, target_dtype: torch.dtype, enable: bool):
    if not enable:
        return
    if not torch.cuda.is_available():
        return
    try:
        if hasattr(p, "transformer") and p.transformer is not None:
            p.transformer.to("cuda", dtype=target_dtype)
    except Exception:
        pass
    try:
        if hasattr(p, "vae") and p.vae is not None:
            p.vae.to("cuda", dtype=target_dtype)
    except Exception:
        pass

# ========= 核心推理/解码逻辑 =========
def _prepare_full_gpu(p, attn_slice):
    if hasattr(p, "reset_device_map"):
        try: p.reset_device_map()
        except Exception: pass
    if not torch.cuda.is_available():
        raise RuntimeError("没有检测到 CUDA")
    runtime_fragmentation_cleanup(synchronize=True, reset_peak=False)
    p.to("cuda")
    if hasattr(p, "enable_attention_slicing"):
        if attn_slice == "auto":
            attn_slice = "max"
        p.enable_attention_slicing(attn_slice)

def _apply_int8_matmul_to_transformer(p, enable: bool):
    try:
        from sdnq.loader import apply_sdnq_options_to_model
    except Exception as e:
        return False, f"未能导入 sdnq.loader: {e}"
    if not hasattr(p, "transformer") or p.transformer is None:
        return False, "pipe.transformer 不存在"
    try:
        p.transformer = apply_sdnq_options_to_model(p.transformer, use_quantized_matmul=bool(enable))
        return True, f"INT8 MatMul = {bool(enable)}"
    except Exception as e:
        return False, f"设置失败: {e}"

# ========= CFG>1 时必须提供 negative_prompt_embeds =========
def _encode_prompt_embeds_then_drop_te(p, prompt_text: str, negative_text: str | None, cfg: float, te_dtype=torch.float32):
    ensure_tokenizer_loaded_fn, ensure_text_encoder_loaded_fn, drop_text_encoder_fn = _require_manager_funcs()

    if not _pipe_accepts_kw(p, "prompt_embeds"):
        raise RuntimeError("pipe 不支持 prompt_embeds")

    ensure_tokenizer_loaded_fn(p)

    te = ensure_text_encoder_loaded_fn(p, device="cuda", dtype=torch.float32)
    try:
        te.to("cuda", dtype=te_dtype)
    except Exception:
        pass

    encode_fn = getattr(p, "encode_prompt", None) or getattr(p, "_encode_prompt", None)
    if encode_fn is None:
        raise RuntimeError("pipe 没有 encode_prompt/_encode_prompt")

    sig = inspect.signature(encode_fn)
    keys = set(sig.parameters.keys())

    def _encode_one(text: str):
        kwargs = {}
        if "prompt" in keys:
            kwargs["prompt"] = text
        if "device" in keys:
            kwargs["device"] = torch.device("cuda")
        if "num_images_per_prompt" in keys:
            kwargs["num_images_per_prompt"] = 1
        if "do_classifier_free_guidance" in keys:
            kwargs["do_classifier_free_guidance"] = False
        if "negative_prompt" in keys:
            kwargs["negative_prompt"] = None

        encoded = encode_fn(**kwargs)

        out = {}
        if isinstance(encoded, torch.Tensor):
            out["prompt_embeds"] = encoded
        elif isinstance(encoded, (tuple, list)):
            out["prompt_embeds"] = encoded[0]
            if len(encoded) >= 3 and isinstance(encoded[2], torch.Tensor):
                out["pooled_prompt_embeds"] = encoded[2]
        elif isinstance(encoded, dict):
            for k in ["prompt_embeds", "pooled_prompt_embeds", "attention_mask"]:
                if k in encoded:
                    out[k] = encoded[k]
        else:
            raise RuntimeError(f"未知 encode 返回类型: {type(encoded)}")
        return out

    do_cfg = bool(cfg is not None and float(cfg) > 1.0)

    pos = _encode_one(prompt_text)
    embeds_kwargs = {"prompt_embeds": pos["prompt_embeds"]}
    if "pooled_prompt_embeds" in pos:
        embeds_kwargs["pooled_prompt_embeds"] = pos["pooled_prompt_embeds"]

    if do_cfg:
        neg_text = negative_text if (negative_text and negative_text.strip()) else ""
        neg = _encode_one(neg_text)
        embeds_kwargs["negative_prompt_embeds"] = neg["prompt_embeds"]
        if "pooled_prompt_embeds" in neg:
            embeds_kwargs["negative_pooled_prompt_embeds"] = neg["pooled_prompt_embeds"]

    drop_text_encoder_fn(p, drop_tokenizer=False)

    for k, v in list(embeds_kwargs.items()):
        if isinstance(v, torch.Tensor):
            embeds_kwargs[k] = v.to("cuda")
    return embeds_kwargs

def _decode_latent_with_pipe_vae(p, latent_t: torch.Tensor, scaling_factor: float, tiled: bool, tile_size):
    if not hasattr(p, "vae") or p.vae is None:
        raise RuntimeError("pipe.vae 不存在")
    vae = p.vae
    vae.to("cuda", dtype=torch.float16)
    lat = latent_t
    if lat.dim() == 3:
        lat = lat.unsqueeze(0)
    lat = lat.to("cuda", dtype=torch.float16)
    with torch.no_grad():
        if tiled and hasattr(vae, "enable_tiling"):
            vae.enable_tiling()
            try:
                vae.tile_sample_size = int(tile_size)
            except Exception:
                pass
        decoded = vae.decode(lat / float(scaling_factor)).sample
    img = (decoded[0] / 2 + 0.5).clamp(0, 1).detach().cpu().permute(1, 2, 0).float().numpy()
    return Image.fromarray((img * 255).astype(np.uint8))

# ========= 图像比例预设 =========
ASPECT_RATIOS = {
    "自定义": None,
    "1:1 方形 (1024×1024)": (1024, 1024),
    "4:3 横屏 (1152×864)": (1152, 864),
    "3:4 竖屏 (864×1152)": (864, 1152),
    "16:9 宽屏 (1344×768)": (1344, 768),
    "9:16 手机屏 (768×1344)": (768, 1344),
    "3:2 照片横 (1216×816)": (1216, 816),
    "2:3 照片竖 (816×1216)": (816, 1216),
    "21:9 超宽 (1536×656)": (1536, 656),
}

def _auto_tile_for_size(w, h):
    m = max(w, h)
    if m >= 1536: return 512
    if m >= 1024: return 768
    return 1024

# ========= Drive 挂载 =========
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# ========= 样式定义 =========
STYLE_CSS = """
<style>
.zimg-panel { font-family: 'Segoe UI', sans-serif; }
.zimg-section {
    background: #1e293b; border-radius: 12px; padding: 16px; margin: 8px 0;
    border: 1px solid #334155;
}
.zimg-section-title {
    color: #38bdf8; font-size: 14px; font-weight: 600;
    margin-bottom: 12px; padding-bottom: 8px;
    border-bottom: 1px solid #334155;
}
.zimg-header {
    background: linear-gradient(135deg, #0ea5e9 0%, #8b5cf6 50%, #ec4899 100%);
    border-radius: 16px; padding: 20px; margin-bottom: 16px;
    box-shadow: 0 4px 20px rgba(14, 165, 233, 0.3);
}
.zimg-header h1 { color: white; font-size: 24px; font-weight: 700; margin: 0 0 4px 0; }
.zimg-header p { color: rgba(255,255,255,0.85); font-size: 12px; margin: 0; }
.zimg-label { color: #94a3b8; font-size: 12px; margin-bottom: 4px; }
.zimg-info { color: #64748b; font-size: 11px; margin-top: 4px; }
</style>
"""

# ========= UI 组件 =========
header_html = widgets.HTML(STYLE_CSS + """
<div class="zimg-panel">
<div class="zimg-header">
    <h1>🎨 Z-Image Turbo</h1>
    <p>TE 编码后卸载 · VAE 常驻显存 · SDPA(Efficient) · TE卸载后碎片整理 · 元数据完整保存</p>
</div>
</div>
""")

allocator_html = widgets.HTML(value=f"""
<div style="margin:10px 0 0 0; padding:10px 12px; border:1px solid #334155; border-radius:12px; background:#0b1220;">
  <div style="font-size:13px; color:#e2e8f0; font-weight:600; margin-bottom:4px;">🧠 Boot/Allocator 状态</div>
  <div style="font-size:13px;">{_allocator_status_text()}</div>
</div>
""")

# ==================== 1. 提示词区域 ====================
prompt_box = widgets.Textarea(
    value="a beautiful cat sitting on a windowsill, soft sunlight, detailed fur, photorealistic",
    placeholder="输入正向提示词...",
    layout=widgets.Layout(width="100%", height="300px"),
)
neg_box = widgets.Textarea(
    value="",
    placeholder="输入负面提示词（CFG≤1无效）...",
    layout=widgets.Layout(width="100%", height="60px"),
)

# ==================== 从 PNG 一键读取元数据并回填 ====================
w_png_meta_path = widgets.Text(
    value="",
    placeholder="填写要读取元数据的 PNG 路径（例如 /content/drive/MyDrive/Z-image/Outputs/img_xxx.png）",
    description="PNG 路径",
    style={'description_width': '90px'},
    layout=widgets.Layout(width="100%")
)

btn_load_meta = widgets.Button(
    description="📥 一键读取 PNG 元数据并回填参数",
    button_style="info",
    layout=widgets.Layout(width="320px", height="36px")
)

btn_copy_meta_json = widgets.Button(
    description="📋 复制元数据 JSON 到剪贴板",
    button_style="",
    layout=widgets.Layout(width="240px", height="36px")
)

meta_status = widgets.HTML(value="<div class='zimg-info'>在此填 PNG 路径后，可一键读取 parameters 元数据并回填面板参数。</div>")

def _read_metadata_from_png(png_path: str) -> dict | None:
    try:
        img = Image.open(png_path)
        if hasattr(img, "info") and "parameters" in img.info:
            return json.loads(img.info["parameters"])
    except Exception as e:
        print(f"读取 PNG 元数据失败: {e}")
    return None

def _safe_float(v, default=None):
    try:
        if v is None: return default
        return float(v)
    except Exception:
        return default

def _safe_int(v, default=None):
    try:
        if v is None: return default
        return int(float(v))
    except Exception:
        return default

def _set_ratio_preset_if_exact_match(width: int, height: int) -> str:
    for name, wh in ASPECT_RATIOS.items():
        if wh is None:
            continue
        if int(wh[0]) == int(width) and int(wh[1]) == int(height):
            return name
    return "自定义"

def _apply_metadata_to_ui(meta: dict):
    if isinstance(meta.get("prompt", None), str):
        prompt_box.value = meta["prompt"]
    if isinstance(meta.get("negative_prompt", None), str):
        neg_box.value = meta["negative_prompt"]

    steps = _safe_int(meta.get("steps", None), default=None)
    if steps is not None:
        w_steps.value = int(max(w_steps.min, min(w_steps.max, steps)))

    cfg = _safe_float(meta.get("cfg_scale", None), default=None)
    if cfg is not None:
        w_cfg.value = float(max(w_cfg.min, min(w_cfg.max, cfg)))

    seed = _safe_int(meta.get("seed", None), default=None)
    if seed is not None:
        w_seed.value = int(seed)

    width = _safe_int(meta.get("width", None), default=None)
    height = _safe_int(meta.get("height", None), default=None)
    if width is not None and height is not None:
        width16 = _round_to_16(int(width))
        height16 = _round_to_16(int(height))
        width16 = int(max(w_width.min, min(w_width.max, width16)))
        height16 = int(max(w_height.min, min(w_height.max, height16)))
        ratio_name = meta.get("aspect_ratio_preset", None)
        if not isinstance(ratio_name, str) or ratio_name not in ASPECT_RATIOS:
            ratio_name = _set_ratio_preset_if_exact_match(width16, height16)
        w_ratio.value = ratio_name
        w_width.value = width16
        w_height.value = height16
        _update_size_info(None)

    svh = meta.get("seed_variance_enhancer", {})
    if isinstance(svh, dict):
        enabled = bool(svh.get("enabled", False))
        w_svh_enable.value = enabled

        rp = _safe_float(svh.get("randomize_percent", None), default=None)
        if rp is not None:
            w_svh_randomize_percent.value = float(max(w_svh_randomize_percent.min, min(w_svh_randomize_percent.max, rp)))

        st = _safe_float(svh.get("strength", None), default=None)
        if st is not None:
            w_svh_strength.value = float(st)

        ni = svh.get("noise_insert", None)
        if isinstance(ni, str) and ni in [o for o in w_svh_noise_insert.options]:
            w_svh_noise_insert.value = ni

        sw = _safe_float(svh.get("steps_switchover_percent", None), default=None)
        if sw is not None:
            w_svh_steps_switchover_percent.value = float(max(w_svh_steps_switchover_percent.min, min(w_svh_steps_switchover_percent.max, sw)))

        ms = svh.get("mask_starts_at", None)
        if isinstance(ms, str) and ms in [o for o in w_svh_mask_starts_at.options]:
            w_svh_mask_starts_at.value = ms

        mp = _safe_float(svh.get("mask_percent", None), default=None)
        if mp is not None:
            w_svh_mask_percent.value = float(max(w_svh_mask_percent.min, min(w_svh_mask_percent.max, mp)))

        sm = svh.get("seed_mode", None)
        if isinstance(sm, str) and sm in [v for _, v in w_svh_seed_mode.options]:
            w_svh_seed_mode.value = sm

        sseed = _safe_int(svh.get("seed", None), default=None)
        if sseed is not None:
            w_svh_seed.value = int(sseed)

    lora = meta.get("lora_slots", {})
    if isinstance(lora, dict):
        def _set_if_in_options(dd: widgets.Dropdown, val: str):
            vals = [v for _, v in dd.options]
            dd.value = val if (val in vals) else ""

        f1 = lora.get("slot1_file", "") or ""
        f2 = lora.get("slot2_file", "") or ""
        f3 = lora.get("slot3_file", "") or ""

        _set_if_in_options(w_lora1, f1)
        _set_if_in_options(w_lora2, f2)
        _set_if_in_options(w_lora3, f3)

        w1 = _safe_float(lora.get("slot1_weight", None), default=None)
        w2 = _safe_float(lora.get("slot2_weight", None), default=None)
        w3 = _safe_float(lora.get("slot3_weight", None), default=None)
        if w1 is not None: w_lora1_w.value = float(max(w_lora1_w.min, min(w_lora1_w.max, w1)))
        if w2 is not None: w_lora2_w.value = float(max(w_lora2_w.min, min(w_lora2_w.max, w2)))
        if w3 is not None: w_lora3_w.value = float(max(w_lora3_w.min, min(w_lora3_w.max, w3)))

def _copy_text_to_clipboard_js(text: str):
    from IPython.display import Javascript, display as _disp
    safe = json.dumps(str(text))
    _disp(Javascript(f"""
    (async () => {{
      try {{
        await navigator.clipboard.writeText({safe});
        console.log("copied");
      }} catch(e) {{
        console.error(e);
      }}
    }})()
    """))

def _on_load_meta(_):
    out.clear_output()
    with out:
        png_path = (w_png_meta_path.value or "").strip()
        if not png_path:
            print("❌ 请先填写 PNG 路径")
            return
        if not os.path.exists(png_path):
            print(f"❌ PNG 不存在: {png_path}")
            return
        meta = _read_metadata_from_png(png_path)
        if not meta:
            print("❌ 未读到 parameters 元数据（该 PNG 可能不是本面板保存的，或没有 parameters 字段）")
            return
        _apply_metadata_to_ui(meta)
        globals()["_last_loaded_png_metadata"] = meta
        meta_status.value = f"<div class='zimg-info'>✅ 已从 PNG 读取并回填：{os.path.basename(png_path)}</div>"
        print("✅ 元数据已回填到面板参数。")
        print("— 可直接点 🚀 生成图像 复现。")
        print("— 如需复制原始 JSON，点「复制元数据 JSON 到剪贴板」。")

def _on_copy_meta_json(_):
    meta = globals().get("_last_loaded_png_metadata", None)
    if not isinstance(meta, dict):
        meta_status.value = "<div class='zimg-info'>⚠️ 尚未读取任何 PNG 元数据；请先点击「一键读取」</div>"
        return
    text = json.dumps(meta, ensure_ascii=False, indent=2)
    _copy_text_to_clipboard_js(text)
    meta_status.value = "<div class='zimg-info'>📋 已将元数据 JSON 复制到剪贴板</div>"

btn_load_meta.on_click(_on_load_meta)
btn_copy_meta_json.on_click(_on_copy_meta_json)

# ==================== 2. 图像尺寸区域 ====================
_current_ratio = None
_updating = False

w_ratio = widgets.Dropdown(
    options=list(ASPECT_RATIOS.keys()),
    value="自定义",
    description="",
    layout=widgets.Layout(width="100%"),
)

w_width = widgets.IntSlider(
    value=1024, min=512, max=2560, step=16,
    description="宽度",
    style={'description_width': '60px'},
    layout=widgets.Layout(width="100%"),
    readout=True,
    continuous_update=False,
)
w_height = widgets.IntSlider(
    value=1024, min=512, max=2560, step=16,
    description="高度",
    style={'description_width': '60px'},
    layout=widgets.Layout(width="100%"),
    readout=True,
    continuous_update=False,
)
w_size_info = widgets.HTML('<div class="zimg-info">📐 最终尺寸: 1024 × 1024 (已对齐16像素)</div>')

def _parse_ratio_from_name(name):
    if name == "自定义":
        return None
    w, h = ASPECT_RATIOS[name]
    return w / h

def _set_initial_size_by_ratio(ratio):
    if ratio is None:
        return 1024, 1024
    if ratio >= 1:
        w = 1024
        h = round(w / ratio)
    else:
        h = 1024
        w = round(h * ratio)
    w = max(512, min(2560, _round_to_16(w)))
    h = max(512, min(2560, _round_to_16(h)))
    return w, h

def _on_ratio_change(change):
    global _current_ratio, _updating
    ratio_name = change["new"]
    ratio = _parse_ratio_from_name(ratio_name)
    _current_ratio = ratio
    if ratio is not None:
        w, h = _set_initial_size_by_ratio(ratio)
        _updating = True
        w_width.value = w
        w_height.value = h
        _updating = False
        _update_size_info(None)
    else:
        _update_size_info(None)

def _on_width_change(change):
    global _current_ratio, _updating
    if _updating or _current_ratio is None:
        _update_size_info(change)
        return
    new_w = _round_to_16(change["new"])
    new_h = _round_to_16(new_w / _current_ratio)
    new_h = max(512, min(2560, new_h))
    _updating = True
    w_height.value = new_h
    _updating = False
    _update_size_info(None)

def _on_height_change(change):
    global _current_ratio, _updating
    if _updating or _current_ratio is None:
        _update_size_info(change)
        return
    new_h = _round_to_16(change["new"])
    new_w = _round_to_16(new_h * _current_ratio)
    new_w = max(512, min(2560, new_w))
    _updating = True
    w_width.value = new_w
    _updating = False
    _update_size_info(None)

def _update_size_info(change):
    w16 = _round_to_16(w_width.value)
    h16 = _round_to_16(w_height.value)
    w_size_info.value = f'<div class="zimg-info">📐 最终尺寸: {w16} × {h16} (已对齐16像素)</div>'

w_ratio.observe(_on_ratio_change, names="value")
w_width.observe(_on_width_change, names="value")
w_height.observe(_on_height_change, names="value")

# ==================== 3. 生成参数区域 ====================
w_steps = widgets.IntSlider(
    value=9, min=2, max=50, step=1,
    description="采样步数",
    style={'description_width': '80px'},
    layout=widgets.Layout(width="100%"),
)
w_cfg = widgets.FloatSlider(
    value=0.0, min=0.0, max=10.0, step=0.1,
    description="CFG 强度",
    style={'description_width': '80px'},
    layout=widgets.Layout(width="100%"),
)
w_seed = widgets.IntText(
    value=-1,
    description="随机种子",
    style={'description_width': '80px'},
    layout=widgets.Layout(width="100%"),
)

# ==================== 3.5 批量张数 ====================
w_batch = widgets.BoundedIntText(
    value=1, min=1, max=200, step=1,
    description="每次张数",
    style={'description_width': '80px'},
    layout=widgets.Layout(width="220px")
)

# ==================== 3.6 Scheduler 选单 ====================
_SCHEDULER_OPTIONS = [
    ("Euler（基线）",                        "euler"),
    ("Euler + Karras",                       "euler_karras"),
    ("Euler + Exponential",                  "euler_exp"),
    ("Euler Ancestral（stochastic）",        "euler_ancestral"),
    ("Euler Ancestral + Karras",             "euler_ancestral_karras"),
    ("DPM++ 2M RF",                          "dpm2m"),
    ("DPM++ 2M RF + Karras",                 "dpm2m_karras"),
    ("DPM++ 2M RF + Exponential",            "dpm2m_exp"),
]

w_scheduler = widgets.Dropdown(
    options=_SCHEDULER_OPTIONS,
    value="euler",
    description="采样器",
    style={'description_width': '80px'},
    layout=widgets.Layout(width="100%"),
)

w_shift = widgets.FloatSlider(
    value=3.0, min=0.5, max=10.0, step=0.1,
    description="shift",
    style={'description_width': '80px'},
    layout=widgets.Layout(width="100%"),
    readout=True,
    continuous_update=False,
)
w_shift_hint = widgets.HTML(
    value="<div class='zimg-info'>💡 shift=3.0 为 Z-Image 默认；值越大构图越受控、细节越少；低步数时影响显著</div>"
)

def _build_scheduler_from_ui(base_config: dict) -> object:
    from diffusers import FlowMatchEulerDiscreteScheduler
    try:
        from diffusers import FlowMatchDPMSolverMultistepScheduler
        _has_dpm_rf = True
    except ImportError:
        _has_dpm_rf = False

    sel = w_scheduler.value
    shift = float(w_shift.value)

    is_dpm = sel.startswith("dpm")
    use_karras = sel.endswith("_karras")
    use_exp    = sel.endswith("_exp")
    stochastic = "ancestral" in sel

    if is_dpm and not _has_dpm_rf:
        print("⚠️ 当前 diffusers 版本不含 FlowMatchDPMSolverMultistepScheduler，已回退到 Euler 基线")
        is_dpm = False

    _euler_supported = {
        "num_train_timesteps", "shift", "use_dynamic_shifting",
        "base_shift", "max_shift", "base_image_seq_len", "max_image_seq_len",
        "invert_sigmas", "use_karras_sigmas", "use_exponential_sigmas",
        "use_beta_sigmas", "time_shift_type", "stochastic_sampling", "shift_terminal",
    }

    cfg = dict(base_config)
    cfg["shift"] = shift
    cfg["use_karras_sigmas"]      = use_karras
    cfg["use_exponential_sigmas"] = use_exp
    cfg["stochastic_sampling"]    = stochastic

    try:
        if is_dpm:
            dpm_cfg = {k: v for k, v in cfg.items()
                       if not k.startswith("_") and k != "stochastic_sampling"}
            sched = FlowMatchDPMSolverMultistepScheduler(**dpm_cfg)
        else:
            euler_cfg = {k: v for k, v in cfg.items()
                         if not k.startswith("_") and k in _euler_supported}
            sched = FlowMatchEulerDiscreteScheduler(**euler_cfg)
        return sched
    except Exception as e:
        print(f"⚠️ Scheduler 构建失败，回退基线: {e}")
        return None

# ==================== 4. 显存优化区域 ====================
w_attn = widgets.Dropdown(
    options=["auto", "max", 1, 2, 4, 8, 16, 32],
    value="auto",
    description="注意力切片",
    style={'description_width': '100px'},
    layout=widgets.Layout(width="100%"),
)
w_int8 = widgets.Checkbox(
    value=True,
    description="启用 INT8 量化加速",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="100%"),
)

w_dtype_mode = widgets.Dropdown(
    options=[
        ("默认 FP32（基线）", "fp32"),
        ("省显存 FP16（推荐）", "fp16"),
        ("省显存 BF16（需GPU支持）", "bf16"),
    ],
    value="fp16",
    description="dtype",
    style={'description_width': '60px'},
    layout=widgets.Layout(width="100%")
)

w_dtype_apply = widgets.Checkbox(
    value=False,
    description="启用 dtype 下沉（transformer/TE 等尽力转为 FP16/BF16）",
    indent=False,
    layout=widgets.Layout(width="100%")
)

dtype_hint = widgets.HTML(
    value="<div class='zimg-info'>💡 默认关闭以不破坏 2048² 基线。</div>"
)

# ==================== 5. VAE 解码区域 ====================
_VAE_SCALING_FACTOR = 0.3611
w_tiled = widgets.Checkbox(
    value=False,
    description="启用 VAE 分块解码（大图推荐）",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="100%")
)
w_tile = widgets.Dropdown(
    options=["auto", 64, 128, 256, 384, 512, 768, 1024],
    value="auto",
    description="分块大小",
    style={'description_width': '80px'},
    layout=widgets.Layout(width="100%")
)

# ==================== 6. 保存选项区域 ====================
w_save_latent = widgets.Checkbox(
    value=False,
    description="保存 Latent (.npz)",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="50%")
)
w_save_png = widgets.Checkbox(
    value=False,
    description="保存 PNG 图像",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="50%")
)
w_latent_dir = widgets.Text(
    value="/content/drive/MyDrive/Z-image/Latent",
    description="Latent 目录",
    style={'description_width': '90px'},
    layout=widgets.Layout(width="100%")
)
w_png_dir = widgets.Text(
    value="/content/drive/MyDrive/Z-image/Outputs",
    description="PNG 目录",
    style={'description_width': '90px'},
    layout=widgets.Layout(width="100%")
)

# ====================
# LoRA 三槽位
# 手动 merge/unmerge（绕开 PEFT，兼容所有 diffusers/peft 版本）
# 单目录：/content/Lora
# ====================
_LORA_DIR = "/content/Lora"

# 当前已 merge 进 transformer 的状态
# { slot_idx: {"file": fn, "weight": w, "sd": {...tensors...}} }
_lora_merged_state: dict = {}

w_lora_dir = widgets.Text(
    value=_LORA_DIR,
    description="LoRA 目录",
    style={'description_width': '90px'},
    layout=widgets.Layout(width="100%")
)

def _list_lora_files_for_ui(folder: str):
    folder = (folder or "").strip()
    if not folder or not os.path.exists(folder):
        return []
    files = []
    for fn in os.listdir(folder):
        if fn.endswith(".safetensors"):
            files.append(fn)
    return sorted(files)

# ---- 手动 merge 核心 ----

def _lora_load_sd(fp: str) -> dict:
    """读取 safetensors，返回原始 state dict（保留在 CPU）"""
    import safetensors.torch as sf
    return sf.load_file(fp, device="cpu")

def _lora_get_transformer(p):
    t = getattr(p, "transformer", None)
    if t is None:
        raise RuntimeError("pipe.transformer 不存在")
    return t

def _lora_build_param_map(transformer) -> dict:
    """构建 transformer 的 named_parameters 映射，去掉 .weight 后缀方便查找"""
    return {k: v for k, v in transformer.named_parameters()}

def _lora_find_param(param_map: dict, base_key: str):
    """
    base_key: LoRA 文件里去掉 .lora_A/B.weight 后的原始 key
    常见前缀变体：diffusion_model.xxx / transformer.xxx / 裸 xxx
    transformer named_parameters 里存的是裸 key（无 diffusion_model. 前缀）
    """
    # 常见前缀，按优先级逐个剥
    prefixes = ["diffusion_model.", "transformer.", "model.diffusion_model.", ""]
    for pfx in prefixes:
        if base_key.startswith(pfx):
            bare = base_key[len(pfx):]
            # 目标 key = bare + ".weight"
            target = bare + ".weight"
            if target in param_map:
                return target, param_map[target]
            # 也试不带 .weight 的（极少数 bias lora）
            if bare in param_map:
                return bare, param_map[bare]
    return None, None

def _lora_compute_delta(sd: dict, base_key: str, scale: float, param_device, param_dtype):
    """
    从 sd 里取 lora_A / lora_B，计算 delta = lora_B @ lora_A * scale
    返回 delta tensor（和参数同设备同 dtype）
    """
    key_A = base_key + ".lora_A.weight"
    key_B = base_key + ".lora_B.weight"

    # 兼容部分 LoRA 用 down/up 命名
    if key_A not in sd:
        key_A = base_key + ".lora_down.weight"
        key_B = base_key + ".lora_up.weight"

    if key_A not in sd or key_B not in sd:
        return None

    A = sd[key_A].to(device=param_device, dtype=torch.float32)
    B = sd[key_B].to(device=param_device, dtype=torch.float32)

    # A shape: (rank, in)   B shape: (out, rank)
    delta = (B @ A) * scale
    return delta.to(dtype=param_dtype)

def _lora_merge_into_transformer(transformer, sd: dict, scale: float) -> int:
    """把一个 LoRA sd merge 进 transformer，返回成功 merge 的层数"""
    param_map = _lora_build_param_map(transformer)

    # 收集所有 base_key（去重）
    base_keys = set()
    for k in sd.keys():
        for suffix in [".lora_A.weight", ".lora_B.weight",
                       ".lora_down.weight", ".lora_up.weight"]:
            if k.endswith(suffix):
                base_keys.add(k[: -len(suffix)])
                break

    merged = 0
    for base_key in base_keys:
        t_key, param = _lora_find_param(param_map, base_key)
        if param is None:
            continue
        delta = _lora_compute_delta(sd, base_key, scale, param.device, param.dtype)
        if delta is None:
            continue
        if delta.shape != param.shape:
            continue
        param.data.add_(delta)
        merged += 1

    return merged

def _lora_unmerge_from_transformer(transformer, sd: dict, scale: float) -> int:
    """把一个 LoRA sd 从 transformer unmerge，返回成功 unmerge 的层数"""
    param_map = _lora_build_param_map(transformer)

    base_keys = set()
    for k in sd.keys():
        for suffix in [".lora_A.weight", ".lora_B.weight",
                       ".lora_down.weight", ".lora_up.weight"]:
            if k.endswith(suffix):
                base_keys.add(k[: -len(suffix)])
                break

    unmerged = 0
    for base_key in base_keys:
        t_key, param = _lora_find_param(param_map, base_key)
        if param is None:
            continue
        delta = _lora_compute_delta(sd, base_key, scale, param.device, param.dtype)
        if delta is None:
            continue
        if delta.shape != param.shape:
            continue
        param.data.sub_(delta)
        unmerged += 1

    return unmerged

def _apply_lora_slots_to_pipe(p, slot_files: list, slot_weights: list, lora_folder: str):
    """
    主入口：对比当前 merge 状态，只做必要的 unmerge / merge。
    slot_files:   长度3，每项是文件名或 None
    slot_weights: 长度3，对应 float 权重
    """
    global _lora_merged_state

    transformer = _lora_get_transformer(p)
    lora_folder = (lora_folder or "").strip()

    for i in range(1, 4):
        fn      = slot_files[i - 1] or None
        weight  = float(slot_weights[i - 1])
        cur     = _lora_merged_state.get(i)

        # 判断是否需要先 unmerge 旧的
        need_unmerge = (
            cur is not None and (
                cur["file"] != fn or
                abs(cur["weight"] - weight) > 1e-6
            )
        )
        if need_unmerge:
            n = _lora_unmerge_from_transformer(transformer, cur["sd"], cur["weight"])
            print(f"   槽位{i} unmerge: {cur['file']} ({n} 层)")
            del _lora_merged_state[i]
            cur = None

        # 判断是否需要 merge 新的
        need_merge = (fn is not None and weight != 0.0 and cur is None)
        if need_merge:
            fp = os.path.join(lora_folder, fn)
            if not os.path.exists(fp):
                raise RuntimeError(f"LoRA 文件不存在: {fp}")
            sd = _lora_load_sd(fp)
            n  = _lora_merge_into_transformer(transformer, sd, weight)
            if n == 0:
                print(f"   ⚠️ 槽位{i}: 未找到任何匹配层，LoRA 可能与模型架构不兼容: {fn}")
            else:
                print(f"   槽位{i} merge: {fn} weight={weight} ({n} 层)")
            _lora_merged_state[i] = {"file": fn, "weight": weight, "sd": sd}

        # fn=None 且 weight=0 且 cur=None：无操作

def _make_lora_slot_ui(slot_idx: int):
    dd = widgets.Dropdown(
        options=[("（不加载）", "")],
        value="",
        description=f"槽位{slot_idx}",
        layout=widgets.Layout(width="100%")
    )
    wt = widgets.FloatSlider(
        value=0.0, min=-2.0, max=2.0, step=0.05,
        description="强度",
        style={'description_width': '60px'},
        layout=widgets.Layout(width="100%"),
        readout=True,
        continuous_update=False,
    )
    return dd, wt

w_lora1, w_lora1_w = _make_lora_slot_ui(1)
w_lora2, w_lora2_w = _make_lora_slot_ui(2)
w_lora3, w_lora3_w = _make_lora_slot_ui(3)

btn_lora_refresh = widgets.Button(
    description="🔄 刷新 LoRA 文件列表",
    button_style="info",
    layout=widgets.Layout(width="220px", height="32px")
)

def _refresh_lora_dropdowns(_=None):
    folder = w_lora_dir.value.strip()
    files = _list_lora_files_for_ui(folder)
    opts = [("（不加载）", "")] + [(fn, fn) for fn in files]

    cur1, cur2, cur3 = w_lora1.value, w_lora2.value, w_lora3.value
    w_lora1.options = opts
    w_lora2.options = opts
    w_lora3.options = opts

    values = [v for _, v in opts]
    w_lora1.value = cur1 if cur1 in values else ""
    w_lora2.value = cur2 if cur2 in values else ""
    w_lora3.value = cur3 if cur3 in values else ""

btn_lora_refresh.on_click(_refresh_lora_dropdowns)
w_lora_dir.observe(lambda c: _refresh_lora_dropdowns(), names="value")
_refresh_lora_dropdowns()

lora_slots_ui = widgets.VBox([
    widgets.HTML('<div class="zimg-label">LoRA 目录（将扫描该目录下所有 .safetensors / .pt / .bin）</div>'),
    w_lora_dir,
    widgets.HTML("<div class='zimg-info'>⚠️ 注意：加载 LoRA 可能导致 2048² 显存不够；可降分辨率继续用。</div>"),
    btn_lora_refresh,
    widgets.VBox([w_lora1, w_lora1_w, w_lora2, w_lora2_w, w_lora3, w_lora3_w])
])

# ==================== 6.6 SeedVarianceEnhancer ====================
w_svh_enable = widgets.Checkbox(
    value=False,
    description="启用 SeedVarianceEnhancer（对 prompt embeds 加噪增强多样性）",
    indent=False,
    layout=widgets.Layout(width="100%")
)
w_svh_randomize_percent = widgets.FloatSlider(
    value=50.0, min=1.0, max=100.0, step=1.0,
    description="randomize%",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%"),
    readout=True,
    continuous_update=False
)
w_svh_strength = widgets.FloatText(
    value=20.0,
    description="strength",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%")
)
w_svh_noise_insert = widgets.Dropdown(
    options=["noise on beginning steps", "noise on ending steps", "noise on all steps", "disabled"],
    value="noise on all steps",
    description="noise_insert",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%")
)
w_svh_steps_switchover_percent = widgets.FloatSlider(
    value=20.0, min=1.0, max=99.0, step=1.0,
    description="switch%",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%"),
    readout=True,
    continuous_update=False
)
w_svh_seed_mode = widgets.Dropdown(
    options=[("跟随出图种子", "follow"), ("自定义 SVH 种子", "custom")],
    value="follow",
    description="SVH seed",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%")
)
w_svh_seed = widgets.IntText(
    value=0,
    description="seed",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%")
)
w_svh_mask_starts_at = widgets.Dropdown(
    options=["beginning", "end"],
    value="beginning",
    description="mask_from",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%")
)
w_svh_mask_percent = widgets.FloatSlider(
    value=0.0, min=0.0, max=99.0, step=1.0,
    description="mask%",
    style={'description_width': '110px'},
    layout=widgets.Layout(width="100%"),
    readout=True,
    continuous_update=False
)
w_svh_log_to_console = widgets.Checkbox(
    value=False,
    description="log_to_console（打印 embedding 统计等）",
    indent=False,
    layout=widgets.Layout(width="100%")
)

def _svh_controls_enabled(enable: bool):
    for w in [
        w_svh_randomize_percent, w_svh_strength, w_svh_noise_insert,
        w_svh_steps_switchover_percent, w_svh_seed_mode, w_svh_seed,
        w_svh_mask_starts_at, w_svh_mask_percent, w_svh_log_to_console
    ]:
        w.disabled = not bool(enable)

def _on_svh_enable_change(change):
    _svh_controls_enabled(change["new"])
    _on_svh_seed_mode_change({"new": w_svh_seed_mode.value})

def _on_svh_seed_mode_change(change):
    w_svh_seed.disabled = (change["new"] != "custom") or (not bool(w_svh_enable.value))

w_svh_enable.observe(_on_svh_enable_change, names="value")
w_svh_seed_mode.observe(_on_svh_seed_mode_change, names="value")
_svh_controls_enabled(w_svh_enable.value)
_on_svh_seed_mode_change({"new": w_svh_seed_mode.value})

def _svh_apply_to_embeds(embeds_kwargs, *, enable, randomize_percent, strength,
                          noise_insert, steps_switchover_percent, seed,
                          mask_starts_at, mask_percent, log_to_console):
    if not enable or noise_insert == "disabled" or strength == 0:
        return embeds_kwargs
    if "prompt_embeds" not in embeds_kwargs or not isinstance(embeds_kwargs["prompt_embeds"], torch.Tensor):
        return embeds_kwargs

    rp = max(1.0, min(100.0, float(randomize_percent))) / 100.0
    mp = max(0.0, min(99.0, float(mask_percent))) / 100.0

    if noise_insert != "noise on all steps" and log_to_console:
        print(f"⚠️ SVH: 当前推理管线不支持按步段切换（{noise_insert}），已按 'noise on all steps' 处理。")

    def _first_null_last_nonnull(seq_t):
        first_null, last_nonnull = -1, -1
        null_seq = [0] * seq_t.size(1)
        for i in range(seq_t.size(1)):
            s = seq_t[:, i, :]
            is_all_zero = bool(torch.all(s == 0).item())
            null_seq[i] = 0 if is_all_zero else 1
            if not is_all_zero: last_nonnull = i
            if is_all_zero and first_null == -1: first_null = i
        return first_null, last_nonnull, null_seq

    def _apply_one(t, local_seed):
        device = t.device
        torch.manual_seed(int(local_seed))
        noise = torch.rand_like(t) * 2 * float(strength) - float(strength)
        torch.manual_seed(int(local_seed) + 1)
        noise_mask = torch.bernoulli(torch.ones_like(t) * rp).bool()

        first_null, last_nonnull, null_seq = _first_null_last_nonnull(t)

        if mp > 0 or (last_nonnull < t.size(1) - 1 and last_nonnull >= 0):
            seq_len = (last_nonnull + 1) if (last_nonnull < t.size(1) - 1 and last_nonnull >= 0) else t.size(1)
            if mask_starts_at == "end":
                mask_start = seq_len - int(seq_len * mp)
                mask_end = t.size(1)
            else:
                mask_start = 0
                mask_end = int(seq_len * mp)
            prompt_mask = torch.arange(t.size(1), device=device).view(1, -1, 1).expand(t.size(0), -1, t.size(2))
            prompt_mask = (prompt_mask >= mask_start) & (prompt_mask < mask_end)
            if first_null > -1:
                null_mask_tensor = ~torch.tensor(null_seq, device=device, dtype=torch.bool)
                null_mask_tensor = null_mask_tensor.view(1, -1, 1).expand(t.size(0), -1, t.size(2))
                prompt_mask = prompt_mask | null_mask_tensor
            noise_mask = noise_mask & (~prompt_mask)

        out = t + (noise * noise_mask)
        if log_to_console:
            with torch.no_grad():
                st = float(torch.std(t).item())
                print(f"SVH: applied noise | seed={local_seed} | rp={rp:.2f} | strength={strength} | std={st:.6f}")
        return out

    embeds_kwargs = dict(embeds_kwargs)
    embeds_kwargs["prompt_embeds"] = _apply_one(embeds_kwargs["prompt_embeds"], seed)
    if "negative_prompt_embeds" in embeds_kwargs and isinstance(embeds_kwargs["negative_prompt_embeds"], torch.Tensor):
        embeds_kwargs["negative_prompt_embeds"] = _apply_one(embeds_kwargs["negative_prompt_embeds"], seed + 999)
    return embeds_kwargs

# ==================== 7. 操作按钮 ====================
btn_run = widgets.Button(
    description="🚀 生成图像",
    button_style="success",
    layout=widgets.Layout(width="160px", height="48px"),
)
btn_clear = widgets.Button(
    description="🗑️ 清空输出",
    button_style="warning",
    layout=widgets.Layout(width="140px", height="48px"),
)

out = widgets.Output(layout=widgets.Layout(
    border="2px solid #334155",
    border_radius="12px",
    padding="16px",
    min_height="300px",
    background="#0f172a",
))

# ==================== 元数据构建与写入 ====================
def _build_generation_metadata(prompt, negative, seed, steps, cfg, width, height,
                                ratio_name, svh_meta, lora_slots, pipe=None,
                                scheduler_name="", scheduler_shift=3.0):
    meta = {
        "prompt": prompt,
        "negative_prompt": negative,
        "seed": int(seed),
        "steps": int(steps),
        "cfg_scale": float(cfg),
        "width": int(width),
        "height": int(height),
        "aspect_ratio_preset": ratio_name,
        "scheduler": scheduler_name,
        "scheduler_shift": float(scheduler_shift),
        "lora_slots": lora_slots or {},
        "seed_variance_enhancer": svh_meta or {},
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    if pipe is not None:
        try: meta["pipeline_class"] = pipe.__class__.__name__
        except Exception: pass
        try:
            if hasattr(pipe, "scheduler") and pipe.scheduler is not None:
                meta["scheduler_class"] = pipe.scheduler.__class__.__name__
        except Exception: pass
        loras = []
        try:
            target = getattr(pipe, "unet", None) or getattr(pipe, "transformer", None)
            if target is not None and hasattr(target, "peft_config") and target.peft_config:
                loras = list(target.peft_config.keys())
        except Exception: pass
        meta["active_lora_adapters"] = loras
    return meta

def _save_latent_with_metadata(latent, metadata, save_path):
    try:
        lat_np = latent.detach().cpu().to(torch.float32).numpy()
        metadata_json = json.dumps(metadata, ensure_ascii=False, indent=2)
        np.savez_compressed(save_path, latent=lat_np,
                            metadata_json=np.array([metadata_json], dtype=object))
        return True
    except Exception as e:
        print(f"❌ 保存 Latent 失败: {e}"); traceback.print_exc(); return False

def _save_png_with_metadata(image, metadata, save_path):
    try:
        from PIL import PngImagePlugin
        pnginfo = PngImagePlugin.PngInfo()
        metadata_json = json.dumps(metadata, ensure_ascii=False, indent=2)
        pnginfo.add_text("parameters", metadata_json)
        pnginfo.add_text("prompt",          str(metadata.get("prompt", "")))
        pnginfo.add_text("negative_prompt", str(metadata.get("negative_prompt", "")))
        pnginfo.add_text("seed",            str(metadata.get("seed", "")))
        pnginfo.add_text("steps",           str(metadata.get("steps", "")))
        pnginfo.add_text("cfg_scale",       str(metadata.get("cfg_scale", "")))
        pnginfo.add_text("width",           str(metadata.get("width", "")))
        pnginfo.add_text("height",          str(metadata.get("height", "")))
        svh = metadata.get("seed_variance_enhancer", {})
        if svh.get("enabled"):
            pnginfo.add_text("svh_enabled",           "true")
            pnginfo.add_text("svh_strength",          str(svh.get("strength", "")))
            pnginfo.add_text("svh_randomize_percent", str(svh.get("randomize_percent", "")))
            pnginfo.add_text("svh_seed",              str(svh.get("seed", "")))
        lora_slots = metadata.get("lora_slots", {})
        for slot_key in ["slot1_file", "slot2_file", "slot3_file"]:
            if lora_slots.get(slot_key):
                pnginfo.add_text(f"lora_{slot_key}", str(lora_slots[slot_key]))
        image.save(save_path, pnginfo=pnginfo)
        return True
    except Exception as e:
        print(f"❌ 保存 PNG 失败: {e}"); traceback.print_exc(); return False

def _read_metadata_from_latent(npz_path):
    try:
        data = np.load(npz_path, allow_pickle=True)
        if "metadata_json" in data:
            json_str = data["metadata_json"]
            if isinstance(json_str, np.ndarray):
                json_str = json_str.item() if json_str.ndim == 0 else json_str[0]
            return json.loads(json_str)
    except Exception as e:
        print(f"读取 latent 元数据失败: {e}")
    return None

# ========= 生成逻辑 =========
def _on_run(_):
    out.clear_output()
    with out:
        sdpa_ctx = None
        _original_scheduler = None
        try:
            p = _get_existing_pipe()
            if p is None:
                print("❌ 未检测到 pipe：请先用模型管理器加载模型")
                return

            prompt      = prompt_box.value.strip()
            negative    = neg_box.value.strip()
            width       = int(w_width.value)
            height      = int(w_height.value)
            steps       = int(w_steps.value)
            cfg         = float(w_cfg.value)
            seed_in     = int(w_seed.value)
            batch_n     = int(w_batch.value)

            attn_slice  = w_attn.value
            int8_enable = bool(w_int8.value)
            scaling     = _VAE_SCALING_FACTOR
            tiled       = bool(w_tiled.value)
            tile_size   = w_tile.value

            save_latent = bool(w_save_latent.value)
            save_png    = bool(w_save_png.value)
            latent_dir  = w_latent_dir.value.strip()
            png_dir     = w_png_dir.value.strip()

            os.makedirs(latent_dir, exist_ok=True)
            os.makedirs(png_dir, exist_ok=True)

            width16  = _round_to_16(width)
            height16 = _round_to_16(height)

            if tile_size == "auto":
                tile_size = _auto_tile_for_size(width16, height16)

            dtype_mode        = str(w_dtype_mode.value)
            target_dtype      = _dtype_from_mode(dtype_mode)
            if target_dtype == torch.bfloat16 and not _bf16_is_supported_cuda():
                print("⚠️ BF16 似乎不支持，自动降级为 FP16")
                target_dtype = torch.float16
            dtype_apply_enabled = bool(w_dtype_apply.value)

            svh_enabled  = bool(w_svh_enable.value)
            svh_seed_mode = str(w_svh_seed_mode.value)

            # -------- Scheduler 替换 --------
            _original_scheduler = p.scheduler
            _base_sched_config  = dict(p.scheduler.config)
            _new_sched = _build_scheduler_from_ui(_base_sched_config)
            if _new_sched is not None:
                p.scheduler = _new_sched
                sched_label = w_scheduler.value
            else:
                sched_label = "euler（回退基线）"

            print("=" * 50)
            print(f"🎯 批量生成: {batch_n} 张")
            print(f"   尺寸: {width16}×{height16} | 步数: {steps} | CFG: {cfg}")
            print(f"   scheduler: {sched_label} | shift: {w_shift.value}")
            print(f"   allocator: {_get_allocator_conf() or '(not set)'}")
            print(f"   dtype_mode={dtype_mode} | apply={dtype_apply_enabled}")
            print("=" * 50)

            if torch.cuda.is_available():
                try: torch.cuda.reset_peak_memory_stats()
                except Exception: pass
                _print_cuda_mem_line("before")

            _prepare_full_gpu(p, attn_slice)
            print(f"✅ GPU 准备完成，注意力切片: {('max' if attn_slice=='auto' else attn_slice)}")

            _maybe_cast_pipe_modules(p, target_dtype=target_dtype, enable=dtype_apply_enabled)

            sdpa_ctx = _enter_sdpa_efficient()
            if sdpa_ctx is not None:
                print("✅ SDPA: EFFICIENT_ATTENTION 优先 已启用")
            else:
                print("⚠️ SDPA: 未能显式启用（将按默认实现/切片运行）")

            ok_int8, msg_int8 = _apply_int8_matmul_to_transformer(p, int8_enable)
            print(("✅ " if ok_int8 else "⚠️ ") + msg_int8)

            # -------- LoRA：批量开始前应用一次 --------
            slot_files   = [w_lora1.value or None, w_lora2.value or None, w_lora3.value or None]
            slot_weights = [float(w_lora1_w.value), float(w_lora2_w.value), float(w_lora3_w.value)]
            lora_folder  = w_lora_dir.value.strip()

            try:
                _apply_lora_slots_to_pipe(p, slot_files=slot_files,
                                          slot_weights=slot_weights, lora_folder=lora_folder)
                active = [(i+1, slot_files[i], slot_weights[i]) for i in range(3) if slot_files[i]]
                if active:
                    print("🎭 LoRA 已应用槽位:")
                    for s, fn, wv in active:
                        print(f"   - 槽位{s}: {fn} | strength={wv}")
                else:
                    print("🎭 LoRA: 未加载（保持基线显存）")
            except Exception as e:
                print("⚠️ LoRA 应用失败（已忽略，不影响无LoRA推理）:", e)
                traceback.print_exc()

            lora_slots_meta = {
                "slot1_file":   slot_files[0] or "",
                "slot1_weight": slot_weights[0],
                "slot2_file":   slot_files[1] or "",
                "slot2_weight": slot_weights[1],
                "slot3_file":   slot_files[2] or "",
                "slot3_weight": slot_weights[2],
                "lora_folder":  lora_folder,
            }

            for i in range(batch_n):
                latent = None
                try:
                    real_seed = int(np.random.randint(0, 2**32 - 1)) if seed_in == -1 else int(seed_in + i)
                    svh_seed  = int(w_svh_seed.value) if (svh_seed_mode == "custom") else int(real_seed)

                    svh_meta = {
                        "enabled":                  svh_enabled,
                        "randomize_percent":        float(w_svh_randomize_percent.value),
                        "strength":                 float(w_svh_strength.value),
                        "noise_insert":             str(w_svh_noise_insert.value),
                        "steps_switchover_percent": float(w_svh_steps_switchover_percent.value),
                        "seed_mode":                str(w_svh_seed_mode.value),
                        "seed":                     int(svh_seed),
                        "mask_starts_at":           str(w_svh_mask_starts_at.value),
                        "mask_percent":             float(w_svh_mask_percent.value),
                    }

                    print("-" * 50)
                    print(f"🖼️ [{i+1}/{batch_n}] seed={real_seed}")

                    t0 = time.time()
                    with torch.inference_mode():
                        te_dtype = target_dtype if dtype_apply_enabled else torch.float16

                        embeds_kwargs = _encode_prompt_embeds_then_drop_te(
                            p, prompt_text=prompt, negative_text=negative, cfg=cfg, te_dtype=te_dtype
                        )
                        embeds_kwargs = _svh_apply_to_embeds(
                            embeds_kwargs,
                            enable=svh_enabled,
                            randomize_percent=float(w_svh_randomize_percent.value),
                            strength=float(w_svh_strength.value),
                            noise_insert=str(w_svh_noise_insert.value),
                            steps_switchover_percent=float(w_svh_steps_switchover_percent.value),
                            seed=int(svh_seed),
                            mask_starts_at=str(w_svh_mask_starts_at.value),
                            mask_percent=float(w_svh_mask_percent.value),
                            log_to_console=bool(w_svh_log_to_console.value),
                        )

                        if torch.cuda.is_available():
                            runtime_fragmentation_cleanup(synchronize=True, reset_peak=True)

                        gen = torch.Generator(device="cpu").manual_seed(real_seed)
                        result = p(
                            prompt=None,
                            **embeds_kwargs,
                            height=height16,
                            width=width16,
                            num_inference_steps=steps,
                            guidance_scale=cfg,
                            generator=gen,
                            output_type="latent",
                        )
                        latent = result.images[0]

                    img = _decode_latent_with_pipe_vae(p, latent, scaling_factor=scaling,
                                                       tiled=tiled, tile_size=tile_size)
                    display(img)

                    metadata = _build_generation_metadata(
                        prompt=prompt, negative=negative, seed=real_seed,
                        steps=steps, cfg=cfg, width=width16, height=height16,
                        ratio_name=w_ratio.value, svh_meta=svh_meta,
                        lora_slots=lora_slots_meta, pipe=p,
                        scheduler_name=sched_label, scheduler_shift=float(w_shift.value),
                    )

                    ts        = datetime.now().strftime("%H%M%S")
                    base_name = f"{real_seed}_{width16}x{height16}_{steps}s_{ts}"

                    if save_latent:
                        latent_path = os.path.join(latent_dir, f"latent_{base_name}.npz")
                        if _save_latent_with_metadata(latent, metadata, latent_path):
                            print(f"💾 Latent: {latent_path}")
                            verify_meta = _read_metadata_from_latent(latent_path)
                            if verify_meta:
                                print(f"   ✓ 元数据验证成功 (seed={verify_meta.get('seed')})")

                    if save_png:
                        png_path = os.path.join(png_dir, f"img_{base_name}.png")
                        if _save_png_with_metadata(img, metadata, png_path):
                            print(f"💾 PNG: {png_path}")
                            verify_meta = _read_metadata_from_png(png_path)
                            if verify_meta:
                                print(f"   ✓ 元数据验证成功 (seed={verify_meta.get('seed')})")

                    dt = time.time() - t0
                    print(f"⏱️ 单张耗时: {dt:.2f} 秒")

                except Exception:
                    traceback.print_exc()
                finally:
                    try: del latent
                    except Exception: pass
                    if torch.cuda.is_available():
                        try: torch.cuda.empty_cache()
                        except Exception: pass
                    gc.collect()

            print("=" * 50)
            print("✨ 批量完成！")
            try:
                _dit = globals().get("_dit_cache_instance", None)
                if _dit is not None and getattr(_dit, "_patched", False):
                    _dit.stats()
            except Exception:
                pass

        except Exception:
            traceback.print_exc()
        finally:
            # scheduler 推理后恢复原始实例
            try:
                if p is not None and _original_scheduler is not None:
                    p.scheduler = _original_scheduler
            except Exception:
                pass
            _exit_sdpa(sdpa_ctx)
            if torch.cuda.is_available():
                try: torch.cuda.empty_cache()
                except Exception: pass
            gc.collect()

def _on_clear(_):
    out.clear_output()

# ==================== UI 布局工具 ====================
def make_section(title_html, *children):
    return widgets.VBox([
        widgets.HTML(f'<div class="zimg-section-title">{title_html}</div>'),
        *children,
    ], layout=widgets.Layout(
        padding="16px", margin="8px 0",
        border="1px solid #334155", border_radius="12px",
        background="#1e293b",
    ))

def make_collapsible_section(title_html, content_widget, open_default=True):
    acc = widgets.Accordion(children=[content_widget])
    acc.set_title(0, title_html)
    acc.selected_index = 0 if open_default else None
    acc.layout = widgets.Layout(margin="8px 0")
    return acc

# ==================== Sections ====================
prompt_inner = widgets.VBox([
    widgets.HTML('<div class="zimg-label">正向提示词</div>'),
    prompt_box,
    widgets.HTML('<div class="zimg-label" style="margin-top:12px;">负面提示词（CFG≤1无效）</div>'),
    neg_box,
    widgets.HTML('<div class="zimg-label" style="margin-top:12px;">📥 一键复制/回填 PNG 元数据</div>'),
    w_png_meta_path,
    widgets.HBox([btn_load_meta, btn_copy_meta_json], layout=widgets.Layout(gap="12px")),
    meta_status,
], layout=widgets.Layout(padding="16px"))

prompt_section = make_collapsible_section("📝 提示词", prompt_inner, open_default=True)

combined_ops_content = widgets.VBox([
    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:8px;">📐 图像尺寸</div>'),
    widgets.HTML('<div class="zimg-label">预设比例（选择后保持比例联动，或选"自定义"自由调整）</div>'),
    w_ratio, w_width, w_height, w_size_info,

    widgets.HTML('<hr style="border-color:#334155;margin:12px 0;">'),

    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:8px;">⚙️ 生成参数</div>'),
    w_steps, w_cfg, w_seed,
    widgets.HTML('<div class="zimg-info">💡 种子 -1 表示随机生成；批量时若非 -1，则使用 seed+i</div>'),
    widgets.HTML('<div class="zimg-label" style="margin-top:10px;">采样器 / Scheduler</div>'),
    w_scheduler, w_shift, w_shift_hint,

    widgets.HTML('<hr style="border-color:#334155;margin:12px 0;">'),

    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:8px;">🎭 LoRA（3槽位·推理前自动应用）</div>'),
    lora_slots_ui,

    widgets.HTML('<hr style="border-color:#334155;margin:12px 0;">'),

    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:8px;">💾 保存设置</div>'),
    widgets.HBox([w_save_latent, w_save_png]),
    w_latent_dir, w_png_dir,
], layout=widgets.Layout(padding="16px"))

combined_ops_section = make_collapsible_section(
    "⚙️ 图像尺寸 / 生成参数 / LoRA / 保存设置",
    combined_ops_content, open_default=True,
)

opt_vae_content = widgets.VBox([
    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:8px;">🚀 显存优化</div>'),
    w_attn, w_int8, w_dtype_mode, w_dtype_apply, dtype_hint,
    widgets.HTML('<div class="zimg-info">💡 注意力切片选 auto 时将按 max 执行（更省显存）</div>'),

    widgets.HTML('<hr style="border-color:#334155;margin:12px 0;">'),

    widgets.HTML('<div class="zimg-section-title" style="margin-bottom:8px;">🖼️ VAE 解码</div>'),
    w_tiled, w_tile,
    widgets.HTML("<div class='zimg-info'>💡 生成大图（≥1536px）建议开启分块解码；tile=256/384/512 更稳｜缩放因子固定 0.3611</div>"),
], layout=widgets.Layout(padding="16px"))

opt_vae_section = make_collapsible_section("🚀 显存优化 / VAE 解码", opt_vae_content, open_default=False)

svh_section_content = make_section(
    "🧬 SeedVarianceEnhancer",
    widgets.HTML("""
    <div style="color:#64748b;font-size:12px;padding:8px;background:#0f172a;border-radius:8px;">
      说明：此处在 <b>TE 编码后</b> 对 <code>prompt_embeds</code>（以及 CFG&gt;1 时的 <code>negative_prompt_embeds</code>）做噪声注入，增强多样性。<br>
      ⚠️ Colab pipeline 模式不支持 ComfyUI conditioning 的"按步 begin/end 切换"，因此 begin/end 会按 all steps 等价处理。
    </div>
    """),
    w_svh_enable, w_svh_randomize_percent, w_svh_strength, w_svh_noise_insert,
    w_svh_steps_switchover_percent, w_svh_mask_starts_at, w_svh_mask_percent,
    w_svh_seed_mode, w_svh_seed, w_svh_log_to_console,
)
svh_section = make_collapsible_section("🧬 SeedVarianceEnhancer（多样性/种子扰动）", svh_section_content, open_default=False)

btn_run.on_click(_on_run)
btn_clear.on_click(_on_clear)

button_section = widgets.HBox(
    [btn_run, w_batch, btn_clear],
    layout=widgets.Layout(justify_content="center", margin="16px 0", gap="16px", align_items="center")
)

main_panel = widgets.VBox([
    header_html,
    allocator_html,
    prompt_section,
    combined_ops_section,
    opt_vae_section,
    svh_section,
    button_section,
    widgets.HTML('<div class="zimg-section-title" style="margin-top:16px;">📺 输出区域</div>'),
    out,
], layout=widgets.Layout(
    width="800px", padding="20px",
    border_radius="16px", background="#0f172a",
))

display(main_panel)